# R NR

## Final version

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy (improved version, includes gene annotation)
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
Also attach gene annotation fields such as gene_name and product

Compatible with the _fully_contained_all_records.csv output format from the R code (with gene annotation)
"""

import os
import pandas as pd
from pathlib import Path
import logging
from collections import defaultdict

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================
# Path configuration
# ============================================
# Input: MTR reads file integrated by R code (_all_records.csv format)
csv_dir = "/path/to/project/results_V3/special/CRC/05.MTR/05.5.MTR_sample_fully_contained"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA_bare"

# Output: filtered results
output_base_dir = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides/R_fully_contained"

# ============================================
# createOutput directory structure
# ============================================
os.makedirs(output_base_dir, exist_ok=True)
logger.info(f"Create output directory: {output_base_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename output by the R code
    format: {sample_name}_xxx_fully_contained_all_records.csv
    text: B1_rna_output_fully_contained_all_records.csv -> B1
    B7_fully_contained_all_records.csv -> B7
    """
    base_name = Path(filename).stem
    # textaftertext "_fully_contained_all_records"
    sample_name = base_name.replace("_fully_contained_all_records", "")
    # onlytext(textfirst), textforMHCfileinsampletextaftertext
    # text: B1_rna_output -> B1
    sample_name = sample_name.split('_')[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    format: CRC.R.{sample_name}.pHLA_II.tsv.gz or CRC.R.{sample_name}.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    mappingtext:     1. id2seq.tsv: Identity -> sseq
    2. sseq2pairs.tsv: sseq -> pairs (textreads ID)
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            logger.error(f" actualcolumn: {id2seq_df.columns.tolist()}")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            logger.error(f" actualcolumn: {sseq2pairs_df.columns.tolist()}")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext:
    source_ids -> Identity (in id2seq) -> sseq -> pairs (in sseq2pairs)
    """
    # Step 1: source_ids -> sseq (passed Identity column)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    # text source_ids mappingto sseq
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)]
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    # text sseq mappingto pairs
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)]
    return pairs_list


def build_read_to_gene_mapping(df_csv):
    """
    build reads ID togeneinformationmapping
    text:
    df_csv: RoutputCSVdata(textQNAMEandgeneannotation column)
    return:
    dict: {QNAME: {gene_info}}
    """
    read_to_gene = {}
    # textavailablegeneannotation column
    gene_cols = ['gene_id', 'gene_name', 'locus_tag', 'product', 'biotype', 'strand', 'gene_length']
    available_cols = [col for col in gene_cols if col in df_csv.columns]
    if not available_cols:
        logger.warning(" warning: texttogeneannotation column, textonlytextQNAME")
        return {}
    logger.info(f" texttogeneannotation column: {', '.join(available_cols)}")
    # foreachQNAMEsavegeneinformation
    # text: textreadtextmappingtotextgenestext
    for _, row in df_csv.iterrows():
        qname = row['QNAME']
        # extractgeneinformation
        gene_info = {}
        for col in available_cols:
            value = row.get(col)
            if pd.notna(value) and value != 'N/A':
                gene_info[col] = value
                # textwithgenetextinformation, textsave
                if 'RNAME' in df_csv.columns:
                    gene_info['chromosome'] = row.get('RNAME')
                    if 'Region_Start' in df_csv.columns and 'Region_End' in df_csv.columns:
                        gene_info['region'] = f"{row.get('Region_Start')}-{row.get('Region_End')}"
                        if 'microbe' in df_csv.columns:
                            gene_info['microbe'] = row.get('microbe')
                            # textreadtextexists, textinformation(textmappingtotext)
                            if qname in read_to_gene:
                                # textgene, texttolist
                                existing = read_to_gene[qname]
                                if isinstance(existing, dict):
                                    read_to_gene[qname] = [existing, gene_info]
                                else:
                                    read_to_gene[qname].append(gene_info)
                            else:
                                read_to_gene[qname] = gene_info
                                logger.info(f" buildtext {len(read_to_gene)} readgenemapping")
                                return read_to_gene


# ============================================
# textprocesstext
# ============================================
def filter_peptides_from_r_output(csv_file, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    fromRtextoutput _all_records.csv fileinextractreads IDandgeneinformation, filterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\n{'='*60}")
    logger.info(f"Process sample: {sample_name}")
    logger.info(f"inputfile: {file_name}")
    logger.info(f"{'='*60}")
    try:
        # readRtextoutputCSVfile
        df_csv = pd.read_csv(csv_file, sep=',')
        logger.info(f" filecolumn: {df_csv.columns.tolist()}")
        logger.info(f" total records: {len(df_csv)}")
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        # statisticsmicrobeinformation
        if 'microbe' in df_csv.columns:
            microbes = df_csv['microbe'].unique()
            logger.info(f" text {len(microbes)} microbes")
            if len(microbes) <= 10:
                logger.info(f" microbe list: {', '.join(microbes)}")
                if n_reads == 0:
                    logger.warning(f" warning: textwithreads ID, skip")
                    return
                # New new: build read -> geneinformation mapping
                logger.info(f"\n buildreadstogenemappingtext...")
                read_to_gene = build_read_to_gene_mapping(df_csv)
                # loadmappingtext
                logger.info(f"\n loadmappingtext...")
                id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
                if id2seq_df is None or sseq2pairs_df is None:
                    logger.error(f" notextloadmappingtext, skipsample {sample_name}")
                    return
                # process MHC Class I
                logger.info(f"\n find MHC Class I textresults...")
                classI_file = find_mhc_file(classI_dir, sample_name, "classI")
                if classI_file:
                    logger.info(f" found Class I file: {os.path.basename(classI_file)}")
                    process_mhc_file(
                        classI_file,                         reads_ids,
                        read_to_gene,
                        id2seq_df,
                        sseq2pairs_df,
                        output_dir,                         sample_name,                         "classI"
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    logger.info(f"\n find MHC Class II textresults...")
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,
            read_to_gene,
            id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII"
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


def process_mhc_file(mhc_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,     output_dir, sample_name, mhc_class):
"""
processMHCtextfile, passedmappingtextfiltermatchedpeptide, textgeneannotation
"""
try:
    # readMHCtextresults
    df_mhc = pd.read_csv(mhc_file, sep='\t')
    # check required columns
    if 'source_ids' not in df_mhc.columns:
        logger.error(f" {mhc_class} filetext source_ids column")
        return
    logger.info(f" MHCtextrecord: {len(df_mhc)} text")
    # text reads_ids textfor set textfind
    reads_ids_set = set(reads_ids)
    # recordmatcheddetailedinformationandgeneinformation
    match_details = []
    gene_annotations = [] # New new: geneannotation columntext
    def check_match_and_get_gene_info(source_ids):
        """
        checkmatchedtextgeneinformation
        """
        if pd.isna(source_ids):
            return False, None
        source_ids_list = str(source_ids).split(',')
        pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
        matched_genes = []
        for pairs in pairs_list:
            if pd.isna(pairs):
                continue
            pairs_str = str(pairs)
            # foundmatchedreadstextgeneinformation
            for read_id in reads_ids_set:
                if read_id in pairs_str:
                    # textreadtextgeneinformation
                    if read_id in read_to_gene:
                        gene_info = read_to_gene[read_id]
                        if isinstance(gene_info, list):
                            matched_genes.extend(gene_info)
                        else:
                            matched_genes.append(gene_info)
                            if matched_genes:
                                return True, matched_genes
                            else:
                                return False, None
                            # filtermatchedrecordtextgeneinformation
                            logger.info(f" startfiltermatchedpeptidetextgeneinformation...")
                            filtered_rows = []
                            for idx, row in df_mhc.iterrows():
                                is_match, gene_infos = check_match_and_get_gene_info(row['source_ids'])
                                if is_match:
                                    filtered_rows.append(row)
                                    # extractgeneannotationinformation
                                    if gene_infos:
                                        # textgenesinformation(textwith)
                                        gene_names = set()
                                        products = set()
                                        microbes = set()
                                        locus_tags = set()
                                        for gene_info in gene_infos:
                                            if 'gene_name' in gene_info:
                                                gene_names.add(str(gene_info['gene_name']))
                                                if 'product' in gene_info:
                                                    products.add(str(gene_info['product']))
                                                    if 'microbe' in gene_info:
                                                        microbes.add(str(gene_info['microbe']))
                                                        if 'locus_tag' in gene_info:
                                                            locus_tags.add(str(gene_info['locus_tag']))
                                                            gene_annotations.append({
                                                                'gene_names': '|'.join(sorted(gene_names)) if gene_names else 'N/A',
                                                                'products': '|'.join(sorted(products)) if products else 'N/A',
                                                                'microbes': '|'.join(sorted(microbes)) if microbes else 'N/A',
                                                                'locus_tags': '|'.join(sorted(locus_tags)) if locus_tags else 'N/A'
                                                            })
                                                        else:
                                                            gene_annotations.append({
                                                                'gene_names': 'N/A',
                                                                'products': 'N/A',
                                                                'microbes': 'N/A',
                                                                'locus_tags': 'N/A'
                                                            })
                                                            if not filtered_rows:
                                                                logger.warning(f" nomatchedrecord, textgenerateOutput files")
                                                                return
                                                            df_mhc_filtered = pd.DataFrame(filtered_rows)
                                                            n_filtered = len(df_mhc_filtered)
                                                            n_total = len(df_mhc)
                                                            percentage = (n_filtered/n_total*100) if n_total > 0 else 0
                                                            logger.info(f" filterresults: {n_filtered}/{n_total} records ({percentage:.2f}%)")
                                                            # New textgeneannotation column
                                                            df_gene_anno = pd.DataFrame(gene_annotations)
                                                            df_mhc_filtered = pd.concat([df_mhc_filtered.reset_index(drop=True),                                                                 df_gene_anno], axis=1)
                                                            # textsampleinformationcolumn
                                                            df_mhc_filtered['sample'] = sample_name
                                                            # textcolumncolumntext, textgeneannotationtextfirsttext
                                                            cols = df_mhc_filtered.columns.tolist()
                                                            priority_cols = ['sample', 'gene_names', 'products', 'microbes', 'locus_tags']
                                                            other_cols = [c for c in cols if c not in priority_cols]
                                                            df_mhc_filtered = df_mhc_filtered[priority_cols + other_cols]
                                                            # savefilterresults
                                                            output_file = os.path.join(output_dir, f"{sample_name}_{mhc_class}_filtered.tsv")
                                                            df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
                                                            logger.info(f" savedtext: {os.path.basename(output_file)}")
                                                            # New new: generategenetextstatistics
                                                            gene_stats = generate_gene_statistics(df_mhc_filtered, mhc_class)
                                                            if not gene_stats.empty:
                                                                gene_stats_file = os.path.join(output_dir,                                                                     f"{sample_name}_{mhc_class}_gene_stats.tsv")
                                                                gene_stats.to_csv(gene_stats_file, sep='\t', index=False)
                                                                logger.info(f" genestatisticssaved: {os.path.basename(gene_stats_file)}")
                                                                # texttopgene
                                                                logger.info(f" Top 5 gene(textpeptide count):")
                                                                for i, row in gene_stats.head(5).iterrows():
                                                                    logger.info(f" {i+1}. {row['gene_name']}: {row['peptide_count']} peptide ({row['product']})")
except Exception as e:
    logger.error(f" process {mhc_class} filetext: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())


def generate_gene_statistics(df_filtered, mhc_class):
    """
    generategenetextpeptidestatistics
    """
    if df_filtered.empty or 'gene_names' not in df_filtered.columns:
        return pd.DataFrame()
    # textgene_names(textgenes, text|text)
    gene_stats = []
    for _, row in df_filtered.iterrows():
        gene_names = str(row['gene_names']).split('|')
        products = str(row['products']).split('|')
        microbes = str(row['microbes']).split('|')
        for i, gene_name in enumerate(gene_names):
            if gene_name != 'N/A':
                product = products[i] if i < len(products) else 'N/A'
                microbe = microbes[i] if i < len(microbes) else 'N/A'
                gene_stats.append({
                    'gene_name': gene_name,
                    'product': product,
                    'microbe': microbe
                })
                if not gene_stats:
                    return pd.DataFrame()
                df_stats = pd.DataFrame(gene_stats)
                # textgenestatisticspeptide count
                gene_summary = df_stats.groupby(['gene_name', 'product', 'microbe']).size().reset_index(name='peptide_count')
                gene_summary = gene_summary.sort_values('peptide_count', ascending=False)
                return gene_summary


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("startfiltertextpeptide (text - textgeneannotation)")
    logger.info("="*60)
    logger.info(f"\ninput directory: {csv_dir}")
    logger.info(f"output directory: {output_base_dir}")
    logger.info(f"\nImprovement notes:")
    logger.info(f" New new: textgeneannotationinformation(gene_name, producttext)")
    logger.info(f" New new: generategenetextpeptidestatistics")
    logger.info(f" New new: textpeptidestextgenes")
    # findall _all_records.csv file
    pattern = "*_fully_contained_all_records.csv"
    csv_files = list(Path(csv_dir).glob(pattern))
    if not csv_files:
        logger.error(f"\nnot foundmatchedfile: {pattern}")
        return
    logger.info(f"\nfound {len(csv_files)} samplesfile")
    # statistics
    stats = {
        'total_samples': len(csv_files),
        'processed_samples': 0,
        'classI_generated': 0,
        'classII_generated': 0,
        'failed_samples': []
    }
    # processtextsamples
    for csv_file in sorted(csv_files):
        try:
            filter_peptides_from_r_output(
                str(csv_file),
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                output_base_dir
)
stats['processed_samples'] += 1
# checkOutput files
sample_name = extract_sample_name(csv_file.name)
classI_output = os.path.join(output_base_dir, f"{sample_name}_classI_filtered.tsv")
classII_output = os.path.join(output_base_dir, f"{sample_name}_classII_filtered.tsv")
if os.path.exists(classI_output):
    stats['classI_generated'] += 1
    if os.path.exists(classII_output):
        stats['classII_generated'] += 1
        except Exception as e:
            logger.error(f"Process samplefailed: {csv_file.name}")
            logger.error(f"error: {str(e)}")
            stats['failed_samples'].append(csv_file.name)
            # textstatisticstext
            logger.info("\n" + "="*60)
            logger.info("Processing completed - statisticstext")
            logger.info("="*60)
            logger.info(f"\ntextsample count: {stats['total_samples']}")
            logger.info(f"processed successfully: {stats['processed_samples']}")
            logger.info(f"generate Class I results: {stats['classI_generated']}")
            logger.info(f"generate Class II results: {stats['classII_generated']}")
            if stats['failed_samples']:
                logger.info(f"\nprocessfailedsample ({len(stats['failed_samples'])}):")
                for sample in stats['failed_samples']:
                    logger.info(f" - {sample}")
                    # savestatistics report
                    summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
                    with open(summary_file, 'w', encoding='utf-8') as f:
                        f.write("textpeptidefilterstatistics report(text - textgeneannotation)\n")
                        f.write("="*60 + "\n\n")
                        f.write(f"input directory: {csv_dir}\n")
                        f.write(f"output directory: {output_base_dir}\n\n")
                        f.write(f"textsample count: {stats['total_samples']}\n")
                        f.write(f"processed successfully: {stats['processed_samples']}\n")
                        f.write(f"generate Class I results: {stats['classI_generated']}\n")
                        f.write(f"generate Class II results: {stats['classII_generated']}\n")
                        if stats['failed_samples']:
                            f.write(f"\nprocessfailedsample:\n")
                            for sample in stats['failed_samples']:
                                f.write(f" - {sample}\n")
                                f.write("\nImprovement notes:\n")
                                f.write(" 1. Output filestextgeneannotation column: gene_names, products, microbes, locus_tags\n")
                                f.write(" 2. Generate gene-level statistics files: [sample]_[class]_gene_stats.tsv\n")
                                f.write(" 3. texttotextpeptidestextgenestext\n")
                                logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                                logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

## Update

### Include boundaries

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
"""

import os
import pandas as pd
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
# ============================================
# Path configuration
# ============================================
# Input: integrated MTR reads files separated by strategy
csv_dir = "/path/to/project/results_V3/special/CRC/05.MTR/05.5.MTR_sample_gene_based"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA_bare"

# Output: filtered results classified by strategy
output_base_dir = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides/R_gene_based"

# Define strategy list
STRATEGIES = [
    "StrategyA_read_based"
    # "StrategyB_fragment_based",
    # "StrategyC_both_ends"
]

# ============================================
# createOutput directory structure
# ============================================
for strategy in STRATEGIES:
    strategy_dir = os.path.join(output_base_dir, strategy)
    os.makedirs(strategy_dir, exist_ok=True)
    logger.info(f"Create directory: {strategy_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename
    textformattext: B1_StrategyA_read_based_combined.csv
    """
    base_name = Path(filename).stem
    parts = base_name.split("_")
    sample_name = parts[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    textformat: CRC.R.B1.pHLA_II.tsv.gz or CRC.R.B1.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    text:
    sample_name: sampletext (text "B7")
    pHLA_bare_dir: pHLA_bare directorypath
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext: source_ids -> Identity -> sseq -> pairs
    text:
    source_ids_list: source_ids list
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    return:
    pairs list (eachtext pairs text)
    """
    # Step 1: source_ids -> sseq (passed Identity)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)] # remove NaN
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)] # remove NaN
    return pairs_list


# ============================================
# textprocesstext
# ============================================
def filter_peptides_by_strategy(csv_file, strategy, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    textreads IDfilterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\nProcess sample: {sample_name} - text: {strategy}")
    try:
        # readMTR readsdata
        df_csv = pd.read_csv(csv_file, sep='\t')
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        if n_reads == 0:
            logger.warning(f" warning: textwithreads ID, skip")
            return
        # loadmappingtext
        id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
        if id2seq_df is None or sseq2pairs_df is None:
            logger.error(f" notextloadmappingtext, skipsample {sample_name}")
            return
        # process MHC Class I
        classI_file = find_mhc_file(classI_dir, sample_name, "classI")
        if classI_file:
            logger.info(f" found Class I file: {os.path.basename(classI_file)}")
            process_mhc_file(
                classI_file,                 reads_ids,                 id2seq_df,
                sseq2pairs_df,
                output_dir,                 sample_name,                 "classI",
                strategy
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,             id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII",
            strategy
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")


def process_mhc_file(mhc_file, reads_ids, id2seq_df, sseq2pairs_df, output_dir, sample_name, mhc_class, strategy):
    """
    processMHCtextfile, passedmappingtextfiltermatchedpeptide
    text:
    mhc_file: MHCtextresultsfilepath
    reads_ids: reads IDtext
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    output_dir: output directory
    sample_name: sampletext
    mhc_class: "classI" or "classII"
    strategy: text
    """
    try:
        # readMHCtextresults
        df_mhc = pd.read_csv(mhc_file, sep='\t')
        # check required columns
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return
        logger.info(f" process {mhc_class}: {len(df_mhc)} records")
        # text reads_ids textfor set textfind
        reads_ids_set = set(reads_ids)
        def check_match_via_mapping(source_ids):
            """
            passedmappingtextchecktextmatched
            source_ids -> Identity -> sseq -> pairs -> check reads ID
            """
            if pd.isna(source_ids):
                return False
            # source_ids text, text
            source_ids_list = str(source_ids).split(',')
            # text pairs
            pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
            # check pairs intext reads_id
            for pairs in pairs_list:
                if pd.isna(pairs):
                    continue
                pairs_str = str(pairs)
                # pairs text
                # checktextwithtext reads_id text pairs in
                if any(read_id in pairs_str for read_id in reads_ids_set):
                    return True
                return False
            # filtermatchedrecord
            df_mhc_filtered = df_mhc[df_mhc['source_ids'].apply(check_match_via_mapping)]
            n_filtered = len(df_mhc_filtered)
            logger.info(f" filterafter: {n_filtered} records ({n_filtered/len(df_mhc)*100:.2f}%)")
            if n_filtered > 0:
                # textandsampleinformationcolumn
                df_mhc_filtered = df_mhc_filtered.copy()
                df_mhc_filtered['sample'] = sample_name
                df_mhc_filtered['strategy'] = strategy
                # savefilterresults
                output_file = os.path.join(
                    output_dir,                     f"{sample_name}_{strategy}_{mhc_class}_filtered.tsv"
)
df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
logger.info(f" savetext: {output_file}")
else:
    logger.warning(f" nomatchedrecord, textgenerateOutput files")
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("starttextfiltertextpeptide")
    logger.info("="*60)
    # statistics
    stats = {strategy: {'samples': 0, 'classI': 0, 'classII': 0}         for strategy in STRATEGIES}
    # text
    for strategy in STRATEGIES:
        logger.info(f"\n{'='*60}")
        logger.info(f"processtext: {strategy}")
        logger.info(f"{'='*60}")
        # findtextallsamplefile
        pattern = f"*_{strategy}_combined.tsv"
        csv_files = list(Path(csv_dir).glob(pattern))
        if not csv_files:
            logger.warning(f"not foundtext {strategy} file")
            continue
        logger.info(f"found {len(csv_files)} samplesfile")
        # output directory
        strategy_output_dir = os.path.join(output_base_dir, strategy)
        # processtextsamples
        for csv_file in csv_files:
            filter_peptides_by_strategy(
                str(csv_file),
                strategy,
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                strategy_output_dir
)
# Updatestatistics
sample_name = extract_sample_name(csv_file.name)
stats[strategy]['samples'] += 1
# checkOutput files
classI_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_classI_filtered.tsv"
)
classII_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_classII_filtered.tsv"
)
if os.path.exists(classI_output):
    stats[strategy]['classI'] += 1
    if os.path.exists(classII_output):
        stats[strategy]['classII'] += 1
        # ============================================
        # textstatisticstext
        # ============================================
        logger.info("\n" + "="*60)
        logger.info("Processing completed - statisticstext")
        logger.info("="*60)
        for strategy in STRATEGIES:
            logger.info(f"\n{strategy}:")
            logger.info(f" Process sampletext: {stats[strategy]['samples']}")
            logger.info(f" generate Class I results: {stats[strategy]['classI']}")
            logger.info(f" generate Class II results: {stats[strategy]['classII']}")
            # savestatistics report
            summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
            with open(summary_file, 'w') as f:
                f.write("textpeptidefilterstatistics report\n")
                f.write("="*60 + "\n\n")
                for strategy in STRATEGIES:
                    f.write(f"{strategy}:\n")
                    f.write(f" Process sampletext: {stats[strategy]['samples']}\n")
                    f.write(f" generate Class I results: {stats[strategy]['classI']}\n")
                    f.write(f" generate Class II results: {stats[strategy]['classII']}\n")
                    f.write("\n")
                    logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                    logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

### textInclude boundaries

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
"""

import os
import pandas as pd
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
# ============================================
# Path configuration
# ============================================
# Input: integrated MTR reads files separated by strategy
csv_dir = "/path/to/project/results_V3/special/CRC/05.MTR/05.5.MTR_sample_gene_based_inner"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA_bare"

# Output: filtered results classified by strategy
output_base_dir = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides/R_gene_based_inner"

# Define strategy list
STRATEGIES = [
    "StrategyA_read_based"
    # "StrategyB_fragment_based",
    # "StrategyC_both_ends"
]

# ============================================
# createOutput directory structure
# ============================================
for strategy in STRATEGIES:
    strategy_dir = os.path.join(output_base_dir, strategy)
    os.makedirs(strategy_dir, exist_ok=True)
    logger.info(f"Create directory: {strategy_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename
    textformattext: B1_StrategyA_read_based_combined.csv
    """
    base_name = Path(filename).stem
    parts = base_name.split("_")
    sample_name = parts[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    textformat: CRC.R.B1.pHLA_II.tsv.gz or CRC.R.B1.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    text:
    sample_name: sampletext (text "B7")
    pHLA_bare_dir: pHLA_bare directorypath
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext: source_ids -> Identity -> sseq -> pairs
    text:
    source_ids_list: source_ids list
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    return:
    pairs list (eachtext pairs text)
    """
    # Step 1: source_ids -> sseq (passed Identity)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)] # remove NaN
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)] # remove NaN
    return pairs_list


# ============================================
# textprocesstext
# ============================================
def filter_peptides_by_strategy(csv_file, strategy, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    textreads IDfilterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\nProcess sample: {sample_name} - text: {strategy}")
    try:
        # readMTR readsdata
        df_csv = pd.read_csv(csv_file, sep = '\t')
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        if n_reads == 0:
            logger.warning(f" warning: textwithreads ID, skip")
            return
        # loadmappingtext
        id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
        if id2seq_df is None or sseq2pairs_df is None:
            logger.error(f" notextloadmappingtext, skipsample {sample_name}")
            return
        # process MHC Class I
        classI_file = find_mhc_file(classI_dir, sample_name, "classI")
        if classI_file:
            logger.info(f" found Class I file: {os.path.basename(classI_file)}")
            process_mhc_file(
                classI_file,                 reads_ids,                 id2seq_df,
                sseq2pairs_df,
                output_dir,                 sample_name,                 "classI",
                strategy
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,             id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII",
            strategy
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")


def process_mhc_file(mhc_file, reads_ids, id2seq_df, sseq2pairs_df, output_dir, sample_name, mhc_class, strategy):
    """
    processMHCtextfile, passedmappingtextfiltermatchedpeptide
    text:
    mhc_file: MHCtextresultsfilepath
    reads_ids: reads IDtext
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    output_dir: output directory
    sample_name: sampletext
    mhc_class: "classI" or "classII"
    strategy: text
    """
    try:
        # readMHCtextresults
        df_mhc = pd.read_csv(mhc_file, sep='\t')
        # check required columns
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return
        logger.info(f" process {mhc_class}: {len(df_mhc)} records")
        # text reads_ids textfor set textfind
        reads_ids_set = set(reads_ids)
        def check_match_via_mapping(source_ids):
            """
            passedmappingtextchecktextmatched
            source_ids -> Identity -> sseq -> pairs -> check reads ID
            """
            if pd.isna(source_ids):
                return False
            # source_ids text, text
            source_ids_list = str(source_ids).split(',')
            # text pairs
            pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
            # check pairs intext reads_id
            for pairs in pairs_list:
                if pd.isna(pairs):
                    continue
                pairs_str = str(pairs)
                # pairs text
                # checktextwithtext reads_id text pairs in
                if any(read_id in pairs_str for read_id in reads_ids_set):
                    return True
                return False
            # filtermatchedrecord
            df_mhc_filtered = df_mhc[df_mhc['source_ids'].apply(check_match_via_mapping)]
            n_filtered = len(df_mhc_filtered)
            logger.info(f" filterafter: {n_filtered} records ({n_filtered/len(df_mhc)*100:.2f}%)")
            if n_filtered > 0:
                # textandsampleinformationcolumn
                df_mhc_filtered = df_mhc_filtered.copy()
                df_mhc_filtered['sample'] = sample_name
                df_mhc_filtered['strategy'] = strategy
                # savefilterresults
                output_file = os.path.join(
                    output_dir,                     f"{sample_name}_{strategy}_{mhc_class}_filtered.tsv"
)
df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
logger.info(f" savetext: {output_file}")
else:
    logger.warning(f" nomatchedrecord, textgenerateOutput files")
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("starttextfiltertextpeptide")
    logger.info("="*60)
    # statistics
    stats = {strategy: {'samples': 0, 'classI': 0, 'classII': 0}         for strategy in STRATEGIES}
    # text
    for strategy in STRATEGIES:
        logger.info(f"\n{'='*60}")
        logger.info(f"processtext: {strategy}")
        logger.info(f"{'='*60}")
        # findtextallsamplefile
        pattern = f"*_{strategy}_combined.tsv"
        csv_files = list(Path(csv_dir).glob(pattern))
        if not csv_files:
            logger.warning(f"not foundtext {strategy} file")
            continue
        logger.info(f"found {len(csv_files)} samplesfile")
        # output directory
        strategy_output_dir = os.path.join(output_base_dir, strategy)
        # processtextsamples
        for csv_file in csv_files:
            filter_peptides_by_strategy(
                str(csv_file),
                strategy,
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                strategy_output_dir
)
# Updatestatistics
sample_name = extract_sample_name(csv_file.name)
stats[strategy]['samples'] += 1
# checkOutput files
classI_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_classI_filtered.tsv"
)
classII_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_classII_filtered.tsv"
)
if os.path.exists(classI_output):
    stats[strategy]['classI'] += 1
    if os.path.exists(classII_output):
        stats[strategy]['classII'] += 1
        # ============================================
        # textstatisticstext
        # ============================================
        logger.info("\n" + "="*60)
        logger.info("Processing completed - statisticstext")
        logger.info("="*60)
        for strategy in STRATEGIES:
            logger.info(f"\n{strategy}:")
            logger.info(f" Process sampletext: {stats[strategy]['samples']}")
            logger.info(f" generate Class I results: {stats[strategy]['classI']}")
            logger.info(f" generate Class II results: {stats[strategy]['classII']}")
            # savestatistics report
            summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
            with open(summary_file, 'w') as f:
                f.write("textpeptidefilterstatistics report\n")
                f.write("="*60 + "\n\n")
                for strategy in STRATEGIES:
                    f.write(f"{strategy}:\n")
                    f.write(f" Process sampletext: {stats[strategy]['samples']}\n")
                    f.write(f" generate Class I results: {stats[strategy]['classI']}\n")
                    f.write(f" generate Class II results: {stats[strategy]['classII']}\n")
                    f.write("\n")
                    logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                    logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

## QUANT

### all

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
"""

import os
import pandas as pd
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
# ============================================
# Path configuration
# ============================================
# Input: integrated MTR reads files separated by strategy
csv_dir = "/path/to/project/results_V3/special/CRC/05.MTR/05.5.MTR_sample_quant_parallel"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA_bare"

# Output: filtered results classified by strategy
output_base_dir = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides_quant_parallel/R_parallel"

# Define strategy list
STRATEGIES = [
    "StrategyA_read_based"
    # "StrategyB_fragment_based",
    # "StrategyC_both_ends"
]

# ============================================
# createOutput directory structure
# ============================================
for strategy in STRATEGIES:
    strategy_dir = os.path.join(output_base_dir, strategy)
    os.makedirs(strategy_dir, exist_ok=True)
    logger.info(f"Create directory: {strategy_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename
    textformattext: B1_StrategyA_read_based_combined.csv
    """
    base_name = Path(filename).stem
    parts = base_name.split("_")
    sample_name = parts[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    textformat: CRC.R.B1.pHLA_II.tsv.gz or CRC.R.B1.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    text:
    sample_name: sampletext (text "B7")
    pHLA_bare_dir: pHLA_bare directorypath
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext: source_ids -> Identity -> sseq -> pairs
    text:
    source_ids_list: source_ids list
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    return:
    pairs list (eachtext pairs text)
    """
    # Step 1: source_ids -> sseq (passed Identity)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)] # remove NaN
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)] # remove NaN
    return pairs_list


# ============================================
# textprocesstext
# ============================================
def filter_peptides_by_strategy(csv_file, strategy, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    textreads IDfilterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\nProcess sample: {sample_name} - text: {strategy}")
    try:
        # readMTR readsdata
        df_csv = pd.read_csv(csv_file)
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        if n_reads == 0:
            logger.warning(f" warning: textwithreads ID, skip")
            return
        # loadmappingtext
        id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
        if id2seq_df is None or sseq2pairs_df is None:
            logger.error(f" notextloadmappingtext, skipsample {sample_name}")
            return
        # process MHC Class I
        classI_file = find_mhc_file(classI_dir, sample_name, "classI")
        if classI_file:
            logger.info(f" found Class I file: {os.path.basename(classI_file)}")
            process_mhc_file(
                classI_file,                 reads_ids,                 id2seq_df,
                sseq2pairs_df,
                output_dir,                 sample_name,                 "classI",
                strategy
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,             id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII",
            strategy
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")


def process_mhc_file(mhc_file, reads_ids, id2seq_df, sseq2pairs_df, output_dir, sample_name, mhc_class, strategy):
    """
    processMHCtextfile, passedmappingtextfiltermatchedpeptide
    text:
    mhc_file: MHCtextresultsfilepath
    reads_ids: reads IDtext
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    output_dir: output directory
    sample_name: sampletext
    mhc_class: "classI" or "classII"
    strategy: text
    """
    try:
        # readMHCtextresults
        df_mhc = pd.read_csv(mhc_file, sep='\t')
        # check required columns
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return
        logger.info(f" process {mhc_class}: {len(df_mhc)} records")
        # text reads_ids textfor set textfind
        reads_ids_set = set(reads_ids)
        def check_match_via_mapping(source_ids):
            """
            passedmappingtextchecktextmatched
            source_ids -> Identity -> sseq -> pairs -> check reads ID
            """
            if pd.isna(source_ids):
                return False
            # source_ids text, text
            source_ids_list = str(source_ids).split(',')
            # text pairs
            pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
            # check pairs intext reads_id
            for pairs in pairs_list:
                if pd.isna(pairs):
                    continue
                pairs_str = str(pairs)
                # pairs text
                # checktextwithtext reads_id text pairs in
                if any(read_id in pairs_str for read_id in reads_ids_set):
                    return True
                return False
            # filtermatchedrecord
            df_mhc_filtered = df_mhc[df_mhc['source_ids'].apply(check_match_via_mapping)]
            n_filtered = len(df_mhc_filtered)
            logger.info(f" filterafter: {n_filtered} records ({n_filtered/len(df_mhc)*100:.2f}%)")
            if n_filtered > 0:
                # textandsampleinformationcolumn
                df_mhc_filtered = df_mhc_filtered.copy()
                df_mhc_filtered['sample'] = sample_name
                df_mhc_filtered['strategy'] = strategy
                # savefilterresults
                output_file = os.path.join(
                    output_dir,                     f"{sample_name}_{strategy}_{mhc_class}_quant_all_filtered.tsv"
)
df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
logger.info(f" savetext: {output_file}")
else:
    logger.warning(f" nomatchedrecord, textgenerateOutput files")
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("starttextfiltertextpeptide")
    logger.info("="*60)
    # statistics
    stats = {strategy: {'samples': 0, 'classI': 0, 'classII': 0}         for strategy in STRATEGIES}
    # text
    for strategy in STRATEGIES:
        logger.info(f"\n{'='*60}")
        logger.info(f"processtext: {strategy}")
        logger.info(f"{'='*60}")
        # findtextallsamplefile
        # pattern = f"*_{strategy}_TARGET_SPECIFIC_combined.csv"
        pattern = f"*_{strategy}_combined.csv"
        csv_files = list(Path(csv_dir).glob(pattern))
        if not csv_files:
            logger.warning(f"not foundtext {strategy} file")
            continue
        logger.info(f"found {len(csv_files)} samplesfile")
        # output directory
        strategy_output_dir = os.path.join(output_base_dir, strategy)
        # processtextsamples
        for csv_file in csv_files:
            filter_peptides_by_strategy(
                str(csv_file),
                strategy,
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                strategy_output_dir
)
# Updatestatistics
sample_name = extract_sample_name(csv_file.name)
stats[strategy]['samples'] += 1
# checkOutput files
classI_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_quant_all_classI_filtered.tsv"
)
classII_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_quant_all_classII_filtered.tsv"
)
if os.path.exists(classI_output):
    stats[strategy]['classI'] += 1
    if os.path.exists(classII_output):
        stats[strategy]['classII'] += 1
        # ============================================
        # textstatisticstext
        # ============================================
        logger.info("\n" + "="*60)
        logger.info("Processing completed - statisticstext")
        logger.info("="*60)
        for strategy in STRATEGIES:
            logger.info(f"\n{strategy}:")
            logger.info(f" Process sampletext: {stats[strategy]['samples']}")
            logger.info(f" generate Class I results: {stats[strategy]['classI']}")
            logger.info(f" generate Class II results: {stats[strategy]['classII']}")
            # savestatistics report
            summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
            with open(summary_file, 'w') as f:
                f.write("textpeptidefilterstatistics report\n")
                f.write("="*60 + "\n\n")
                for strategy in STRATEGIES:
                    f.write(f"{strategy}:\n")
                    f.write(f" Process sampletext: {stats[strategy]['samples']}\n")
                    f.write(f" generate Class I results: {stats[strategy]['classI']}\n")
                    f.write(f" generate Class II results: {stats[strategy]['classII']}\n")
                    f.write("\n")
                    logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                    logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

### specific

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
"""

import os
import pandas as pd
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
# ============================================
# Path configuration
# ============================================
# Input: integrated MTR reads files separated by strategy
csv_dir = "/path/to/project/results_V3/special/CRC/05.MTR/05.5.MTR_sample_quant_parallel"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/pHLA_bare"

# Output: filtered results classified by strategy
output_base_dir = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides_quant_parallel/R_parallel"

# Define strategy list
STRATEGIES = [
    "StrategyA_read_based"
    # "StrategyB_fragment_based",
    # "StrategyC_both_ends"
]

# ============================================
# createOutput directory structure
# ============================================
for strategy in STRATEGIES:
    strategy_dir = os.path.join(output_base_dir, strategy)
    os.makedirs(strategy_dir, exist_ok=True)
    logger.info(f"Create directory: {strategy_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename
    textformattext: B1_StrategyA_read_based_combined.csv
    """
    base_name = Path(filename).stem
    parts = base_name.split("_")
    sample_name = parts[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    textformat: CRC.R.B1.pHLA_II.tsv.gz or CRC.R.B1.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    text:
    sample_name: sampletext (text "B7")
    pHLA_bare_dir: pHLA_bare directorypath
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext: source_ids -> Identity -> sseq -> pairs
    text:
    source_ids_list: source_ids list
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    return:
    pairs list (eachtext pairs text)
    """
    # Step 1: source_ids -> sseq (passed Identity)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)] # remove NaN
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)] # remove NaN
    return pairs_list


# ============================================
# textprocesstext
# ============================================
def filter_peptides_by_strategy(csv_file, strategy, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    textreads IDfilterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\nProcess sample: {sample_name} - text: {strategy}")
    try:
        # readMTR readsdata
        df_csv = pd.read_csv(csv_file)
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        if n_reads == 0:
            logger.warning(f" warning: textwithreads ID, skip")
            return
        # loadmappingtext
        id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
        if id2seq_df is None or sseq2pairs_df is None:
            logger.error(f" notextloadmappingtext, skipsample {sample_name}")
            return
        # process MHC Class I
        classI_file = find_mhc_file(classI_dir, sample_name, "classI")
        if classI_file:
            logger.info(f" found Class I file: {os.path.basename(classI_file)}")
            process_mhc_file(
                classI_file,                 reads_ids,                 id2seq_df,
                sseq2pairs_df,
                output_dir,                 sample_name,                 "classI",
                strategy
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,             id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII",
            strategy
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")


def process_mhc_file(mhc_file, reads_ids, id2seq_df, sseq2pairs_df, output_dir, sample_name, mhc_class, strategy):
    """
    processMHCtextfile, passedmappingtextfiltermatchedpeptide
    text:
    mhc_file: MHCtextresultsfilepath
    reads_ids: reads IDtext
    id2seq_df: id2seq mappingtext
    sseq2pairs_df: sseq2pairs mappingtext
    output_dir: output directory
    sample_name: sampletext
    mhc_class: "classI" or "classII"
    strategy: text
    """
    try:
        # readMHCtextresults
        df_mhc = pd.read_csv(mhc_file, sep='\t')
        # check required columns
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return
        logger.info(f" process {mhc_class}: {len(df_mhc)} records")
        # text reads_ids textfor set textfind
        reads_ids_set = set(reads_ids)
        def check_match_via_mapping(source_ids):
            """
            passedmappingtextchecktextmatched
            source_ids -> Identity -> sseq -> pairs -> check reads ID
            """
            if pd.isna(source_ids):
                return False
            # source_ids text, text
            source_ids_list = str(source_ids).split(',')
            # text pairs
            pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
            # check pairs intext reads_id
            for pairs in pairs_list:
                if pd.isna(pairs):
                    continue
                pairs_str = str(pairs)
                # pairs text
                # checktextwithtext reads_id text pairs in
                if any(read_id in pairs_str for read_id in reads_ids_set):
                    return True
                return False
            # filtermatchedrecord
            df_mhc_filtered = df_mhc[df_mhc['source_ids'].apply(check_match_via_mapping)]
            n_filtered = len(df_mhc_filtered)
            logger.info(f" filterafter: {n_filtered} records ({n_filtered/len(df_mhc)*100:.2f}%)")
            if n_filtered > 0:
                # textandsampleinformationcolumn
                df_mhc_filtered = df_mhc_filtered.copy()
                df_mhc_filtered['sample'] = sample_name
                df_mhc_filtered['strategy'] = strategy
                # savefilterresults
                output_file = os.path.join(
                    output_dir,                     f"{sample_name}_{strategy}_{mhc_class}_filtered.tsv"
)
df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
logger.info(f" savetext: {output_file}")
else:
    logger.warning(f" nomatchedrecord, textgenerateOutput files")
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("starttextfiltertextpeptide")
    logger.info("="*60)
    # statistics
    stats = {strategy: {'samples': 0, 'classI': 0, 'classII': 0}         for strategy in STRATEGIES}
    # text
    for strategy in STRATEGIES:
        logger.info(f"\n{'='*60}")
        logger.info(f"processtext: {strategy}")
        logger.info(f"{'='*60}")
        # findtextallsamplefile
        pattern = f"*_{strategy}_TARGET_SPECIFIC_combined.csv"
        # pattern = f"*_{strategy}_combined.csv"
        csv_files = list(Path(csv_dir).glob(pattern))
        if not csv_files:
            logger.warning(f"not foundtext {strategy} file")
            continue
        logger.info(f"found {len(csv_files)} samplesfile")
        # output directory
        strategy_output_dir = os.path.join(output_base_dir, strategy)
        # processtextsamples
        for csv_file in csv_files:
            filter_peptides_by_strategy(
                str(csv_file),
                strategy,
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                strategy_output_dir
)
# Updatestatistics
sample_name = extract_sample_name(csv_file.name)
stats[strategy]['samples'] += 1
# checkOutput files
classI_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_quant_specific_classI_filtered.tsv"
)
classII_output = os.path.join(
    strategy_output_dir,     f"{sample_name}_{strategy}_quant_specific_all_classII_filtered.tsv"
)
if os.path.exists(classI_output):
    stats[strategy]['classI'] += 1
    if os.path.exists(classII_output):
        stats[strategy]['classII'] += 1
        # ============================================
        # textstatisticstext
        # ============================================
        logger.info("\n" + "="*60)
        logger.info("Processing completed - statisticstext")
        logger.info("="*60)
        for strategy in STRATEGIES:
            logger.info(f"\n{strategy}:")
            logger.info(f" Process sampletext: {stats[strategy]['samples']}")
            logger.info(f" generate Class I results: {stats[strategy]['classI']}")
            logger.info(f" generate Class II results: {stats[strategy]['classII']}")
            # savestatistics report
            summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
            with open(summary_file, 'w') as f:
                f.write("textpeptidefilterstatistics report\n")
                f.write("="*60 + "\n\n")
                for strategy in STRATEGIES:
                    f.write(f"{strategy}:\n")
                    f.write(f" Process sampletext: {stats[strategy]['samples']}\n")
                    f.write(f" generate Class I results: {stats[strategy]['classI']}\n")
                    f.write(f" generate Class II results: {stats[strategy]['classII']}\n")
                    f.write("\n")
                    logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                    logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

## Check 15 peptides

In [ ]:
"""
Check whether the TSV file contains target peptide sequences
"""

import pandas as pd

# readTSVfile
input_file = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides/R_fully_contained_gtf/B7_classI_filtered.tsv"

print(f"readfile: {input_file}")
df = pd.read_csv(input_file, sep='\t')

print(f"\nfileinformation:")
print(f" total rows: {len(df)}")
print(f" column names: {list(df.columns)}")

# checkpeptidecolumntextexists
if 'peptide' not in df.columns:
    print(f"\nFailed error: fileintextwith'peptide'column!")
    print(f"availablecolumn: {list(df.columns)}")
    exit(1)

# Peptide list of interest
target_peptides = [
    "DLDKAYKNF",
    "DPKRLTFR",
    "EIHSLLMER",
    "FGGPDGGDG",
    "FRGAKGGYV",
    "FTNSDYFSGR",
    "FVDAPLFTA",
    "GGDGGKGGD",
    "GPDGGDGGK",
    "NALLRLEKAR",
    "NSDYFSGRI",
    "SGDREIGFT",
    "VHSHITKPV",
    "YRDYDFTNS",
    "YSKNRKHGYR"
]

print(f"\n{'='*70}")
print(f"check {len(target_peptides)} targettextsequence")
print(f"{'='*70}")

# checkeachtargettextexists
found_peptides = []
not_found_peptides = []

for peptide in target_peptides:
    if peptide in df['peptide'].values:
        found_peptides.append(peptide)
        # textallrecord
        matches = df[df['peptide'] == peptide]
        print(f"\nOK found: {peptide}")
        print(f" text: {len(matches)}")
        # statisticsBindLeveltext
        if 'BindLevel' in df.columns:
            bindlevel_counts = matches['BindLevel'].value_counts(dropna=False)
            print(f" BindLeveltext:")
            # textstatistics
            no_bind = bindlevel_counts.get(float('nan'), 0) if pd.isna(bindlevel_counts.index).any() else 0
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            # processtext(NaN)
            null_count = matches['BindLevel'].isna().sum()
            print(f" notextandtext (text): {null_count} text")
            print(f" text (<= WB): {weak_bind} text")
            print(f" text (<= SB): {strong_bind} text")
            # textallBindLeveltext(text)
            print(f" detailedBindLevellist:")
            for idx, row in matches.iterrows():
                bind_val = row['BindLevel']
                if pd.isna(bind_val):
                    print(f" - text (notextandtext)")
                else:
                    print(f" - {bind_val}")
            else:
                not_found_peptides.append(peptide)

# textresults
print(f"\n{'='*70}")
print(f"checkresultstext")
print(f"{'='*70}")
print(f"foundtextsequence: {len(found_peptides)}/{len(target_peptides)}")
print(f"not foundtextsequence: {len(not_found_peptides)}/{len(target_peptides)}")

if found_peptides:
    print(f"\nOK foundtext ({len(found_peptides)}):")
    for peptide in found_peptides:
        matches = df[df['peptide'] == peptide]
        count = len(matches)
        # statisticsBindLevel
        if 'BindLevel' in df.columns:
            null_count = matches['BindLevel'].isna().sum()
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            print(f" - {peptide} (text {count} text): notextandtext={null_count}, WB={weak_bind}, SB={strong_bind}")
        else:
            print(f" - {peptide} (text {count} text)")

if not_found_peptides:
    print(f"\nFailed not foundtext ({len(not_found_peptides)}):")
    for peptide in not_found_peptides:
        print(f" - {peptide}")

print(f"\ncompleted!")

## QUANT-CHECK

### all

In [ ]:
"""
Check whether the TSV file contains target peptide sequences
"""
import pandas as pd

# readTSVfile - Updatefortextoutputpath
input_file = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides_quant_parallel/R_parallel/StrategyA_read_based/B7_StrategyA_read_based_classI_quant_all_filtered.tsv"

print(f"readfile: {input_file}")
df = pd.read_csv(input_file, sep='\t')

print(f"\nfileinformation:")
print(f" total rows: {len(df)}")
print(f" column names: {list(df.columns)}")

# checkpeptidecolumntextexists
if 'peptide' not in df.columns:
    print(f"\nFailed error: fileintextwith'peptide'column!")
    print(f"availablecolumn: {list(df.columns)}")
    exit(1)

# Peptide list of interest
target_peptides = [
    "DLDKAYKNF",
    "DPKRLTFR",
    "EIHSLLMER",
    "FGGPDGGDG",
    "FRGAKGGYV",
    "FTNSDYFSGR",
    "FVDAPLFTA",
    "GGDGGKGGD",
    "GPDGGDGGK",
    "NALLRLEKAR",
    "NSDYFSGRI",
    "SGDREIGFT",
    "VHSHITKPV",
    "YRDYDFTNS",
    "YSKNRKHGYR"
]

print(f"\n{'='*70}")
print(f"check {len(target_peptides)} targettextsequence")
print(f"{'='*70}")

# checkeachtargettextexists
found_peptides = []
not_found_peptides = []

for peptide in target_peptides:
    if peptide in df['peptide'].values:
        found_peptides.append(peptide)
        # textallrecord
        matches = df[df['peptide'] == peptide]
        print(f"\nOK found: {peptide}")
        print(f" text: {len(matches)}")
        # statisticsBindLeveltext
        if 'BindLevel' in df.columns:
            bindlevel_counts = matches['BindLevel'].value_counts(dropna=False)
            print(f" BindLeveltext:")
            # textstatistics
            no_bind = bindlevel_counts.get(float('nan'), 0) if pd.isna(bindlevel_counts.index).any() else 0
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            # processtext(NaN)
            null_count = matches['BindLevel'].isna().sum()
            print(f" notextandtext (text): {null_count} text")
            print(f" text (<= WB): {weak_bind} text")
            print(f" text (<= SB): {strong_bind} text")
            # textallBindLeveltext(text)
            print(f" detailedBindLevellist:")
            for idx, row in matches.iterrows():
                bind_val = row['BindLevel']
                if pd.isna(bind_val):
                    print(f" - text (notextandtext)")
                else:
                    print(f" - {bind_val}")
            else:
                not_found_peptides.append(peptide)

# textresults
print(f"\n{'='*70}")
print(f"checkresultstext")
print(f"{'='*70}")
print(f"foundtextsequence: {len(found_peptides)}/{len(target_peptides)}")
print(f"not foundtextsequence: {len(not_found_peptides)}/{len(target_peptides)}")

if found_peptides:
    print(f"\nOK foundtext ({len(found_peptides)}):")
    for peptide in found_peptides:
        matches = df[df['peptide'] == peptide]
        count = len(matches)
        # statisticsBindLevel
        if 'BindLevel' in df.columns:
            null_count = matches['BindLevel'].isna().sum()
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            print(f" - {peptide} (text {count} text): notextandtext={null_count}, WB={weak_bind}, SB={strong_bind}")
        else:
            print(f" - {peptide} (text {count} text)")

if not_found_peptides:
    print(f"\nFailed not foundtext ({len(not_found_peptides)}):")
    for peptide in not_found_peptides:
        print(f" - {peptide}")

print(f"\ncompleted!")

### specific

In [ ]:
"""
Check whether the TSV file contains target peptide sequences
"""
import pandas as pd

# readTSVfile - Updatefortextoutputpath
input_file = "/path/to/project/results_V3/special/CRC/06.antigen/06.2.MTR_peptides_quant_parallel/R_parallel/StrategyA_read_based/B7_StrategyA_read_based_classI_filtered.tsv"

print(f"readfile: {input_file}")
df = pd.read_csv(input_file, sep='\t')

print(f"\nfileinformation:")
print(f" total rows: {len(df)}")
print(f" column names: {list(df.columns)}")

# checkpeptidecolumntextexists
if 'peptide' not in df.columns:
    print(f"\nFailed error: fileintextwith'peptide'column!")
    print(f"availablecolumn: {list(df.columns)}")
    exit(1)

# Peptide list of interest
target_peptides = [
    "DLDKAYKNF",
    "DPKRLTFR",
    "EIHSLLMER",
    "FGGPDGGDG",
    "FRGAKGGYV",
    "FTNSDYFSGR",
    "FVDAPLFTA",
    "GGDGGKGGD",
    "GPDGGDGGK",
    "NALLRLEKAR",
    "NSDYFSGRI",
    "SGDREIGFT",
    "VHSHITKPV",
    "YRDYDFTNS",
    "YSKNRKHGYR"
]

print(f"\n{'='*70}")
print(f"check {len(target_peptides)} targettextsequence")
print(f"{'='*70}")

# checkeachtargettextexists
found_peptides = []
not_found_peptides = []

for peptide in target_peptides:
    if peptide in df['peptide'].values:
        found_peptides.append(peptide)
        # textallrecord
        matches = df[df['peptide'] == peptide]
        print(f"\nOK found: {peptide}")
        print(f" text: {len(matches)}")
        # statisticsBindLeveltext
        if 'BindLevel' in df.columns:
            bindlevel_counts = matches['BindLevel'].value_counts(dropna=False)
            print(f" BindLeveltext:")
            # textstatistics
            no_bind = bindlevel_counts.get(float('nan'), 0) if pd.isna(bindlevel_counts.index).any() else 0
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            # processtext(NaN)
            null_count = matches['BindLevel'].isna().sum()
            print(f" notextandtext (text): {null_count} text")
            print(f" text (<= WB): {weak_bind} text")
            print(f" text (<= SB): {strong_bind} text")
            # textallBindLeveltext(text)
            print(f" detailedBindLevellist:")
            for idx, row in matches.iterrows():
                bind_val = row['BindLevel']
                if pd.isna(bind_val):
                    print(f" - text (notextandtext)")
                else:
                    print(f" - {bind_val}")
            else:
                not_found_peptides.append(peptide)

# textresults
print(f"\n{'='*70}")
print(f"checkresultstext")
print(f"{'='*70}")
print(f"foundtextsequence: {len(found_peptides)}/{len(target_peptides)}")
print(f"not foundtextsequence: {len(not_found_peptides)}/{len(target_peptides)}")

if found_peptides:
    print(f"\nOK foundtext ({len(found_peptides)}):")
    for peptide in found_peptides:
        matches = df[df['peptide'] == peptide]
        count = len(matches)
        # statisticsBindLevel
        if 'BindLevel' in df.columns:
            null_count = matches['BindLevel'].isna().sum()
            weak_bind = matches[matches['BindLevel'] == '<= WB'].shape[0]
            strong_bind = matches[matches['BindLevel'] == '<= SB'].shape[0]
            print(f" - {peptide} (text {count} text): notextandtext={null_count}, WB={weak_bind}, SB={strong_bind}")
        else:
            print(f" - {peptide} (text {count} text)")

if not_found_peptides:
    print(f"\nFailed not foundtext ({len(not_found_peptides)}):")
    for peptide in not_found_peptides:
        print(f" - {peptide}")

print(f"\ncompleted!")

# MS

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy (improved version, includes gene annotation)
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
Also attach gene annotation fields such as gene_name and product

Compatible with the _fully_contained_all_records.csv output format from the R code (with gene annotation)
"""

import os
import pandas as pd
from pathlib import Path
import logging
from collections import defaultdict

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================
# Path configuration
# ============================================
# Input: MTR reads file integrated by R code (_all_records.csv format)
csv_dir = "/path/to/project/results_V3/special/CRC_MS/05.MTR/05.5.MTR_sample_fully_contained"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA_bare"

# Output: filtered results
output_base_dir = "/path/to/project/results_V3/special/CRC_MS/06.antigen/06.2.MTR_peptides/R_fully_contained"

# ============================================
# createOutput directory structure
# ============================================
os.makedirs(output_base_dir, exist_ok=True)
logger.info(f"Create output directory: {output_base_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename output by the R code
    format: {sample_name}_xxx_fully_contained_all_records.csv
    text: B1_rna_output_fully_contained_all_records.csv -> B1
    B7_fully_contained_all_records.csv -> B7
    """
    base_name = Path(filename).stem
    # textaftertext "_fully_contained_all_records"
    sample_name = base_name.replace("_fully_contained_all_records", "")
    # onlytext(textfirst), textforMHCfileinsampletextaftertext
    # text: B1_rna_output -> B1
    sample_name = sample_name.split('_')[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    format: CRC.R.{sample_name}.pHLA_II.tsv.gz or CRC.R.{sample_name}.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    mappingtext:     1. id2seq.tsv: Identity -> sseq
    2. sseq2pairs.tsv: sseq -> pairs (textreads ID)
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            logger.error(f" actualcolumn: {id2seq_df.columns.tolist()}")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            logger.error(f" actualcolumn: {sseq2pairs_df.columns.tolist()}")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext:
    source_ids -> Identity (in id2seq) -> sseq -> pairs (in sseq2pairs)
    """
    # Step 1: source_ids -> sseq (passed Identity column)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    # text source_ids mappingto sseq
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)]
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    # text sseq mappingto pairs
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)]
    return pairs_list


def build_read_to_gene_mapping(df_csv):
    """
    build reads ID togeneinformationmapping
    text:
    df_csv: RoutputCSVdata(textQNAMEandgeneannotation column)
    return:
    dict: {QNAME: {gene_info}}
    """
    read_to_gene = {}
    # textavailablegeneannotation column
    gene_cols = ['gene_id', 'gene_name', 'locus_tag', 'product', 'biotype', 'strand', 'gene_length']
    available_cols = [col for col in gene_cols if col in df_csv.columns]
    if not available_cols:
        logger.warning(" warning: texttogeneannotation column, textonlytextQNAME")
        return {}
    logger.info(f" texttogeneannotation column: {', '.join(available_cols)}")
    # foreachQNAMEsavegeneinformation
    # text: textreadtextmappingtotextgenestext
    for _, row in df_csv.iterrows():
        qname = row['QNAME']
        # extractgeneinformation
        gene_info = {}
        for col in available_cols:
            value = row.get(col)
            if pd.notna(value) and value != 'N/A':
                gene_info[col] = value
                # textwithgenetextinformation, textsave
                if 'RNAME' in df_csv.columns:
                    gene_info['chromosome'] = row.get('RNAME')
                    if 'Region_Start' in df_csv.columns and 'Region_End' in df_csv.columns:
                        gene_info['region'] = f"{row.get('Region_Start')}-{row.get('Region_End')}"
                        if 'microbe' in df_csv.columns:
                            gene_info['microbe'] = row.get('microbe')
                            # textreadtextexists, textinformation(textmappingtotext)
                            if qname in read_to_gene:
                                # textgene, texttolist
                                existing = read_to_gene[qname]
                                if isinstance(existing, dict):
                                    read_to_gene[qname] = [existing, gene_info]
                                else:
                                    read_to_gene[qname].append(gene_info)
                            else:
                                read_to_gene[qname] = gene_info
                                logger.info(f" buildtext {len(read_to_gene)} readgenemapping")
                                return read_to_gene


# ============================================
# textprocesstext
# ============================================
def filter_peptides_from_r_output(csv_file, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    fromRtextoutput _all_records.csv fileinextractreads IDandgeneinformation, filterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\n{'='*60}")
    logger.info(f"Process sample: {sample_name}")
    logger.info(f"inputfile: {file_name}")
    logger.info(f"{'='*60}")
    try:
        # readRtextoutputCSVfile
        df_csv = pd.read_csv(csv_file, sep=',')
        logger.info(f" filecolumn: {df_csv.columns.tolist()}")
        logger.info(f" total records: {len(df_csv)}")
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        # statisticsmicrobeinformation
        if 'microbe' in df_csv.columns:
            microbes = df_csv['microbe'].unique()
            logger.info(f" text {len(microbes)} microbes")
            if len(microbes) <= 10:
                logger.info(f" microbe list: {', '.join(microbes)}")
                if n_reads == 0:
                    logger.warning(f" warning: textwithreads ID, skip")
                    return
                # New new: build read -> geneinformation mapping
                logger.info(f"\n buildreadstogenemappingtext...")
                read_to_gene = build_read_to_gene_mapping(df_csv)
                # loadmappingtext
                logger.info(f"\n loadmappingtext...")
                id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
                if id2seq_df is None or sseq2pairs_df is None:
                    logger.error(f" notextloadmappingtext, skipsample {sample_name}")
                    return
                # process MHC Class I
                logger.info(f"\n find MHC Class I textresults...")
                classI_file = find_mhc_file(classI_dir, sample_name, "classI")
                if classI_file:
                    logger.info(f" found Class I file: {os.path.basename(classI_file)}")
                    process_mhc_file(
                        classI_file,                         reads_ids,
                        read_to_gene,
                        id2seq_df,
                        sseq2pairs_df,
                        output_dir,                         sample_name,                         "classI"
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    logger.info(f"\n find MHC Class II textresults...")
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,
            read_to_gene,
            id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII"
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


def process_mhc_file(mhc_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,     output_dir, sample_name, mhc_class):
"""
processMHCtextfile, passedmappingtextfiltermatchedpeptide, textgeneannotation
"""
try:
    # readMHCtextresults
    df_mhc = pd.read_csv(mhc_file, sep='\t')
    # check required columns
    if 'source_ids' not in df_mhc.columns:
        logger.error(f" {mhc_class} filetext source_ids column")
        return
    logger.info(f" MHCtextrecord: {len(df_mhc)} text")
    # text reads_ids textfor set textfind
    reads_ids_set = set(reads_ids)
    # recordmatcheddetailedinformationandgeneinformation
    match_details = []
    gene_annotations = [] # New new: geneannotation columntext
    def check_match_and_get_gene_info(source_ids):
        """
        checkmatchedtextgeneinformation
        """
        if pd.isna(source_ids):
            return False, None
        source_ids_list = str(source_ids).split(',')
        pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
        matched_genes = []
        for pairs in pairs_list:
            if pd.isna(pairs):
                continue
            pairs_str = str(pairs)
            # foundmatchedreadstextgeneinformation
            for read_id in reads_ids_set:
                if read_id in pairs_str:
                    # textreadtextgeneinformation
                    if read_id in read_to_gene:
                        gene_info = read_to_gene[read_id]
                        if isinstance(gene_info, list):
                            matched_genes.extend(gene_info)
                        else:
                            matched_genes.append(gene_info)
                            if matched_genes:
                                return True, matched_genes
                            else:
                                return False, None
                            # filtermatchedrecordtextgeneinformation
                            logger.info(f" startfiltermatchedpeptidetextgeneinformation...")
                            filtered_rows = []
                            for idx, row in df_mhc.iterrows():
                                is_match, gene_infos = check_match_and_get_gene_info(row['source_ids'])
                                if is_match:
                                    filtered_rows.append(row)
                                    # extractgeneannotationinformation
                                    if gene_infos:
                                        # textgenesinformation(textwith)
                                        gene_names = set()
                                        products = set()
                                        microbes = set()
                                        locus_tags = set()
                                        for gene_info in gene_infos:
                                            if 'gene_name' in gene_info:
                                                gene_names.add(str(gene_info['gene_name']))
                                                if 'product' in gene_info:
                                                    products.add(str(gene_info['product']))
                                                    if 'microbe' in gene_info:
                                                        microbes.add(str(gene_info['microbe']))
                                                        if 'locus_tag' in gene_info:
                                                            locus_tags.add(str(gene_info['locus_tag']))
                                                            gene_annotations.append({
                                                                'gene_names': '|'.join(sorted(gene_names)) if gene_names else 'N/A',
                                                                'products': '|'.join(sorted(products)) if products else 'N/A',
                                                                'microbes': '|'.join(sorted(microbes)) if microbes else 'N/A',
                                                                'locus_tags': '|'.join(sorted(locus_tags)) if locus_tags else 'N/A'
                                                            })
                                                        else:
                                                            gene_annotations.append({
                                                                'gene_names': 'N/A',
                                                                'products': 'N/A',
                                                                'microbes': 'N/A',
                                                                'locus_tags': 'N/A'
                                                            })
                                                            if not filtered_rows:
                                                                logger.warning(f" nomatchedrecord, textgenerateOutput files")
                                                                return
                                                            df_mhc_filtered = pd.DataFrame(filtered_rows)
                                                            n_filtered = len(df_mhc_filtered)
                                                            n_total = len(df_mhc)
                                                            percentage = (n_filtered/n_total*100) if n_total > 0 else 0
                                                            logger.info(f" filterresults: {n_filtered}/{n_total} records ({percentage:.2f}%)")
                                                            # New textgeneannotation column
                                                            df_gene_anno = pd.DataFrame(gene_annotations)
                                                            df_mhc_filtered = pd.concat([df_mhc_filtered.reset_index(drop=True),                                                                 df_gene_anno], axis=1)
                                                            # textsampleinformationcolumn
                                                            df_mhc_filtered['sample'] = sample_name
                                                            # textcolumncolumntext, textgeneannotationtextfirsttext
                                                            cols = df_mhc_filtered.columns.tolist()
                                                            priority_cols = ['sample', 'gene_names', 'products', 'microbes', 'locus_tags']
                                                            other_cols = [c for c in cols if c not in priority_cols]
                                                            df_mhc_filtered = df_mhc_filtered[priority_cols + other_cols]
                                                            # savefilterresults
                                                            output_file = os.path.join(output_dir, f"{sample_name}_{mhc_class}_filtered.tsv")
                                                            df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
                                                            logger.info(f" savedtext: {os.path.basename(output_file)}")
                                                            # New new: generategenetextstatistics
                                                            gene_stats = generate_gene_statistics(df_mhc_filtered, mhc_class)
                                                            if not gene_stats.empty:
                                                                gene_stats_file = os.path.join(output_dir,                                                                     f"{sample_name}_{mhc_class}_gene_stats.tsv")
                                                                gene_stats.to_csv(gene_stats_file, sep='\t', index=False)
                                                                logger.info(f" genestatisticssaved: {os.path.basename(gene_stats_file)}")
                                                                # texttopgene
                                                                logger.info(f" Top 5 gene(textpeptide count):")
                                                                for i, row in gene_stats.head(5).iterrows():
                                                                    logger.info(f" {i+1}. {row['gene_name']}: {row['peptide_count']} peptide ({row['product']})")
except Exception as e:
    logger.error(f" process {mhc_class} filetext: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())


def generate_gene_statistics(df_filtered, mhc_class):
    """
    generategenetextpeptidestatistics
    """
    if df_filtered.empty or 'gene_names' not in df_filtered.columns:
        return pd.DataFrame()
    # textgene_names(textgenes, text|text)
    gene_stats = []
    for _, row in df_filtered.iterrows():
        gene_names = str(row['gene_names']).split('|')
        products = str(row['products']).split('|')
        microbes = str(row['microbes']).split('|')
        for i, gene_name in enumerate(gene_names):
            if gene_name != 'N/A':
                product = products[i] if i < len(products) else 'N/A'
                microbe = microbes[i] if i < len(microbes) else 'N/A'
                gene_stats.append({
                    'gene_name': gene_name,
                    'product': product,
                    'microbe': microbe
                })
                if not gene_stats:
                    return pd.DataFrame()
                df_stats = pd.DataFrame(gene_stats)
                # textgenestatisticspeptide count
                gene_summary = df_stats.groupby(['gene_name', 'product', 'microbe']).size().reset_index(name='peptide_count')
                gene_summary = gene_summary.sort_values('peptide_count', ascending=False)
                return gene_summary


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("startfiltertextpeptide (text - textgeneannotation)")
    logger.info("="*60)
    logger.info(f"\ninput directory: {csv_dir}")
    logger.info(f"output directory: {output_base_dir}")
    logger.info(f"\nImprovement notes:")
    logger.info(f" New new: textgeneannotationinformation(gene_name, producttext)")
    logger.info(f" New new: generategenetextpeptidestatistics")
    logger.info(f" New new: textpeptidestextgenes")
    # findall _all_records.csv file
    pattern = "*_fully_contained_all_records.csv"
    csv_files = list(Path(csv_dir).glob(pattern))
    if not csv_files:
        logger.error(f"\nnot foundmatchedfile: {pattern}")
        return
    logger.info(f"\nfound {len(csv_files)} samplesfile")
    # statistics
    stats = {
        'total_samples': len(csv_files),
        'processed_samples': 0,
        'classI_generated': 0,
        'classII_generated': 0,
        'failed_samples': []
    }
    # processtextsamples
    for csv_file in sorted(csv_files):
        try:
            filter_peptides_from_r_output(
                str(csv_file),
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                output_base_dir
)
stats['processed_samples'] += 1
# checkOutput files
sample_name = extract_sample_name(csv_file.name)
classI_output = os.path.join(output_base_dir, f"{sample_name}_classI_filtered.tsv")
classII_output = os.path.join(output_base_dir, f"{sample_name}_classII_filtered.tsv")
if os.path.exists(classI_output):
    stats['classI_generated'] += 1
    if os.path.exists(classII_output):
        stats['classII_generated'] += 1
        except Exception as e:
            logger.error(f"Process samplefailed: {csv_file.name}")
            logger.error(f"error: {str(e)}")
            stats['failed_samples'].append(csv_file.name)
            # textstatisticstext
            logger.info("\n" + "="*60)
            logger.info("Processing completed - statisticstext")
            logger.info("="*60)
            logger.info(f"\ntextsample count: {stats['total_samples']}")
            logger.info(f"processed successfully: {stats['processed_samples']}")
            logger.info(f"generate Class I results: {stats['classI_generated']}")
            logger.info(f"generate Class II results: {stats['classII_generated']}")
            if stats['failed_samples']:
                logger.info(f"\nprocessfailedsample ({len(stats['failed_samples'])}):")
                for sample in stats['failed_samples']:
                    logger.info(f" - {sample}")
                    # savestatistics report
                    summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
                    with open(summary_file, 'w', encoding='utf-8') as f:
                        f.write("textpeptidefilterstatistics report(text - textgeneannotation)\n")
                        f.write("="*60 + "\n\n")
                        f.write(f"input directory: {csv_dir}\n")
                        f.write(f"output directory: {output_base_dir}\n\n")
                        f.write(f"textsample count: {stats['total_samples']}\n")
                        f.write(f"processed successfully: {stats['processed_samples']}\n")
                        f.write(f"generate Class I results: {stats['classI_generated']}\n")
                        f.write(f"generate Class II results: {stats['classII_generated']}\n")
                        if stats['failed_samples']:
                            f.write(f"\nprocessfailedsample:\n")
                            for sample in stats['failed_samples']:
                                f.write(f" - {sample}\n")
                                f.write("\nImprovement notes:\n")
                                f.write(" 1. Output filestextgeneannotation column: gene_names, products, microbes, locus_tags\n")
                                f.write(" 2. Generate gene-level statistics files: [sample]_[class]_gene_stats.tsv\n")
                                f.write(" 3. texttotextpeptidestextgenestext\n")
                                logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                                logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

# CRC 88 parires

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Filter antigen peptides by strategy (improved version, includes gene annotation)
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
Also attach gene annotation fields such as gene_name and product

Compatible with the _fully_contained_all_records.csv output format from the R code (with gene annotation)
"""

import os
import pandas as pd
from pathlib import Path
import logging
from collections import defaultdict

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================
# Path configuration
# ============================================
# Input: MTR reads file integrated by R code (_all_records.csv format)
csv_dir = "/path/to/project/results_V3/CRC/05.MTR_88pairs/05.5.MTR_sample_fully_contained"

# Input: MHC binding prediction results
classI_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA/HLAI"
classII_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA/HLAII"

# Input: directory containing mapping relationship files
pHLA_bare_dir = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_inhouse/v2/pHLA_bare"

# Output: filtered results
output_base_dir = "/path/to/project/results_V3/special/CRC_MS/06.antigen/06.2.MTR_peptides/R_fully_contained"

# ============================================
# createOutput directory structure
# ============================================
os.makedirs(output_base_dir, exist_ok=True)
logger.info(f"Create output directory: {output_base_dir}")

# ============================================
# Helper functions
# ============================================
def extract_sample_name(filename):
    """
    Extract the sample name from the filename output by the R code
    format: {sample_name}_xxx_fully_contained_all_records.csv
    text: B1_rna_output_fully_contained_all_records.csv -> B1
    B7_fully_contained_all_records.csv -> B7
    """
    base_name = Path(filename).stem
    # textaftertext "_fully_contained_all_records"
    sample_name = base_name.replace("_fully_contained_all_records", "")
    # onlytext(textfirst), textforMHCfileinsampletextaftertext
    # text: B1_rna_output -> B1
    sample_name = sample_name.split('_')[0]
    return sample_name


def find_mhc_file(mhc_dir, sample_name, mhc_class):
    """
    findMHCtextfile
    format: CRC.R.{sample_name}.pHLA_II.tsv.gz or CRC.R.{sample_name}.pHLA_I.tsv.gz
    """
    if mhc_class == "classI":
        pattern = f"*.{sample_name}.pHLA_I.tsv*"
    else: # classII
        pattern = f"*.{sample_name}.pHLA_II.tsv*"
        mhc_files = list(Path(mhc_dir).glob(pattern))
        if mhc_files:
            if len(mhc_files) > 1:
                logger.warning(f" foundtext {mhc_class} file, text: {mhc_files[0]}")
                return str(mhc_files[0])
            return None


def load_mapping_tables(sample_name, pHLA_bare_dir):
    """
    loadmappingtext
    mappingtext:     1. id2seq.tsv: Identity -> sseq
    2. sseq2pairs.tsv: sseq -> pairs (textreads ID)
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    """
    # buildsampledirectorypath, formattext: CRC.R.B7
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(pHLA_bare_dir).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    sample_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {sample_dir.name}")
    # buildmappingfilepath
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(sample_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(sample_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = pd.read_csv(id2seq_file, sep='\t')
        sseq2pairs_df = pd.read_csv(sseq2pairs_file, sep='\t')
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            logger.error(f" actualcolumn: {id2seq_df.columns.tolist()}")
            return None, None
        if 'sseq' not in sseq2pairs_df.columns or 'pairs' not in sseq2pairs_df.columns:
            logger.error(f" sseq2pairs file is missing required columns (sseq, pairs)")
            logger.error(f" actualcolumn: {sseq2pairs_df.columns.tolist()}")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext:
    source_ids -> Identity (in id2seq) -> sseq -> pairs (in sseq2pairs)
    """
    # Step 1: source_ids -> sseq (passed Identity column)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    # text source_ids mappingto sseq
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)]
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    # text sseq mappingto pairs
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)]
    return pairs_list


def build_read_to_gene_mapping(df_csv):
    """
    build reads ID togeneinformationmapping
    text:
    df_csv: RoutputCSVdata(textQNAMEandgeneannotation column)
    return:
    dict: {QNAME: {gene_info}}
    """
    read_to_gene = {}
    # textavailablegeneannotation column
    gene_cols = ['gene_id', 'gene_name', 'locus_tag', 'product', 'biotype', 'strand', 'gene_length']
    available_cols = [col for col in gene_cols if col in df_csv.columns]
    if not available_cols:
        logger.warning(" warning: texttogeneannotation column, textonlytextQNAME")
        return {}
    logger.info(f" texttogeneannotation column: {', '.join(available_cols)}")
    # foreachQNAMEsavegeneinformation
    # text: textreadtextmappingtotextgenestext
    for _, row in df_csv.iterrows():
        qname = row['QNAME']
        # extractgeneinformation
        gene_info = {}
        for col in available_cols:
            value = row.get(col)
            if pd.notna(value) and value != 'N/A':
                gene_info[col] = value
                # textwithgenetextinformation, textsave
                if 'RNAME' in df_csv.columns:
                    gene_info['chromosome'] = row.get('RNAME')
                    if 'Region_Start' in df_csv.columns and 'Region_End' in df_csv.columns:
                        gene_info['region'] = f"{row.get('Region_Start')}-{row.get('Region_End')}"
                        if 'microbe' in df_csv.columns:
                            gene_info['microbe'] = row.get('microbe')
                            # textreadtextexists, textinformation(textmappingtotext)
                            if qname in read_to_gene:
                                # textgene, texttolist
                                existing = read_to_gene[qname]
                                if isinstance(existing, dict):
                                    read_to_gene[qname] = [existing, gene_info]
                                else:
                                    read_to_gene[qname].append(gene_info)
                            else:
                                read_to_gene[qname] = gene_info
                                logger.info(f" buildtext {len(read_to_gene)} readgenemapping")
                                return read_to_gene


# ============================================
# textprocesstext
# ============================================
def filter_peptides_from_r_output(csv_file, classI_dir, classII_dir, pHLA_bare_dir, output_dir):
    """
    fromRtextoutput _all_records.csv fileinextractreads IDandgeneinformation, filterpeptidetextresults
    """
    # extractsampletext
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\n{'='*60}")
    logger.info(f"Process sample: {sample_name}")
    logger.info(f"inputfile: {file_name}")
    logger.info(f"{'='*60}")
    try:
        # readRtextoutputCSVfile
        df_csv = pd.read_csv(csv_file, sep=',')
        logger.info(f" filecolumn: {df_csv.columns.tolist()}")
        logger.info(f" total records: {len(df_csv)}")
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        # statisticsmicrobeinformation
        if 'microbe' in df_csv.columns:
            microbes = df_csv['microbe'].unique()
            logger.info(f" text {len(microbes)} microbes")
            if len(microbes) <= 10:
                logger.info(f" microbe list: {', '.join(microbes)}")
                if n_reads == 0:
                    logger.warning(f" warning: textwithreads ID, skip")
                    return
                # New new: build read -> geneinformation mapping
                logger.info(f"\n buildreadstogenemappingtext...")
                read_to_gene = build_read_to_gene_mapping(df_csv)
                # loadmappingtext
                logger.info(f"\n loadmappingtext...")
                id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, pHLA_bare_dir)
                if id2seq_df is None or sseq2pairs_df is None:
                    logger.error(f" notextloadmappingtext, skipsample {sample_name}")
                    return
                # process MHC Class I
                logger.info(f"\n find MHC Class I textresults...")
                classI_file = find_mhc_file(classI_dir, sample_name, "classI")
                if classI_file:
                    logger.info(f" found Class I file: {os.path.basename(classI_file)}")
                    process_mhc_file(
                        classI_file,                         reads_ids,
                        read_to_gene,
                        id2seq_df,
                        sseq2pairs_df,
                        output_dir,                         sample_name,                         "classI"
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    # process MHC Class II
    logger.info(f"\n find MHC Class II textresults...")
    classII_file = find_mhc_file(classII_dir, sample_name, "classII")
    if classII_file:
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        process_mhc_file(
            classII_file,             reads_ids,
            read_to_gene,
            id2seq_df,
            sseq2pairs_df,
            output_dir,             sample_name,             "classII"
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())


def process_mhc_file(mhc_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,     output_dir, sample_name, mhc_class):
"""
processMHCtextfile, passedmappingtextfiltermatchedpeptide, textgeneannotation
"""
try:
    # readMHCtextresults
    df_mhc = pd.read_csv(mhc_file, sep='\t')
    # check required columns
    if 'source_ids' not in df_mhc.columns:
        logger.error(f" {mhc_class} filetext source_ids column")
        return
    logger.info(f" MHCtextrecord: {len(df_mhc)} text")
    # text reads_ids textfor set textfind
    reads_ids_set = set(reads_ids)
    # recordmatcheddetailedinformationandgeneinformation
    match_details = []
    gene_annotations = [] # New new: geneannotation columntext
    def check_match_and_get_gene_info(source_ids):
        """
        checkmatchedtextgeneinformation
        """
        if pd.isna(source_ids):
            return False, None
        source_ids_list = str(source_ids).split(',')
        pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
        matched_genes = []
        for pairs in pairs_list:
            if pd.isna(pairs):
                continue
            pairs_str = str(pairs)
            # foundmatchedreadstextgeneinformation
            for read_id in reads_ids_set:
                if read_id in pairs_str:
                    # textreadtextgeneinformation
                    if read_id in read_to_gene:
                        gene_info = read_to_gene[read_id]
                        if isinstance(gene_info, list):
                            matched_genes.extend(gene_info)
                        else:
                            matched_genes.append(gene_info)
                            if matched_genes:
                                return True, matched_genes
                            else:
                                return False, None
                            # filtermatchedrecordtextgeneinformation
                            logger.info(f" startfiltermatchedpeptidetextgeneinformation...")
                            filtered_rows = []
                            for idx, row in df_mhc.iterrows():
                                is_match, gene_infos = check_match_and_get_gene_info(row['source_ids'])
                                if is_match:
                                    filtered_rows.append(row)
                                    # extractgeneannotationinformation
                                    if gene_infos:
                                        # textgenesinformation(textwith)
                                        gene_names = set()
                                        products = set()
                                        microbes = set()
                                        locus_tags = set()
                                        for gene_info in gene_infos:
                                            if 'gene_name' in gene_info:
                                                gene_names.add(str(gene_info['gene_name']))
                                                if 'product' in gene_info:
                                                    products.add(str(gene_info['product']))
                                                    if 'microbe' in gene_info:
                                                        microbes.add(str(gene_info['microbe']))
                                                        if 'locus_tag' in gene_info:
                                                            locus_tags.add(str(gene_info['locus_tag']))
                                                            gene_annotations.append({
                                                                'gene_names': '|'.join(sorted(gene_names)) if gene_names else 'N/A',
                                                                'products': '|'.join(sorted(products)) if products else 'N/A',
                                                                'microbes': '|'.join(sorted(microbes)) if microbes else 'N/A',
                                                                'locus_tags': '|'.join(sorted(locus_tags)) if locus_tags else 'N/A'
                                                            })
                                                        else:
                                                            gene_annotations.append({
                                                                'gene_names': 'N/A',
                                                                'products': 'N/A',
                                                                'microbes': 'N/A',
                                                                'locus_tags': 'N/A'
                                                            })
                                                            if not filtered_rows:
                                                                logger.warning(f" nomatchedrecord, textgenerateOutput files")
                                                                return
                                                            df_mhc_filtered = pd.DataFrame(filtered_rows)
                                                            n_filtered = len(df_mhc_filtered)
                                                            n_total = len(df_mhc)
                                                            percentage = (n_filtered/n_total*100) if n_total > 0 else 0
                                                            logger.info(f" filterresults: {n_filtered}/{n_total} records ({percentage:.2f}%)")
                                                            # New textgeneannotation column
                                                            df_gene_anno = pd.DataFrame(gene_annotations)
                                                            df_mhc_filtered = pd.concat([df_mhc_filtered.reset_index(drop=True),                                                                 df_gene_anno], axis=1)
                                                            # textsampleinformationcolumn
                                                            df_mhc_filtered['sample'] = sample_name
                                                            # textcolumncolumntext, textgeneannotationtextfirsttext
                                                            cols = df_mhc_filtered.columns.tolist()
                                                            priority_cols = ['sample', 'gene_names', 'products', 'microbes', 'locus_tags']
                                                            other_cols = [c for c in cols if c not in priority_cols]
                                                            df_mhc_filtered = df_mhc_filtered[priority_cols + other_cols]
                                                            # savefilterresults
                                                            output_file = os.path.join(output_dir, f"{sample_name}_{mhc_class}_filtered.tsv")
                                                            df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
                                                            logger.info(f" savedtext: {os.path.basename(output_file)}")
                                                            # New new: generategenetextstatistics
                                                            gene_stats = generate_gene_statistics(df_mhc_filtered, mhc_class)
                                                            if not gene_stats.empty:
                                                                gene_stats_file = os.path.join(output_dir,                                                                     f"{sample_name}_{mhc_class}_gene_stats.tsv")
                                                                gene_stats.to_csv(gene_stats_file, sep='\t', index=False)
                                                                logger.info(f" genestatisticssaved: {os.path.basename(gene_stats_file)}")
                                                                # texttopgene
                                                                logger.info(f" Top 5 gene(textpeptide count):")
                                                                for i, row in gene_stats.head(5).iterrows():
                                                                    logger.info(f" {i+1}. {row['gene_name']}: {row['peptide_count']} peptide ({row['product']})")
except Exception as e:
    logger.error(f" process {mhc_class} filetext: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())


def generate_gene_statistics(df_filtered, mhc_class):
    """
    generategenetextpeptidestatistics
    """
    if df_filtered.empty or 'gene_names' not in df_filtered.columns:
        return pd.DataFrame()
    # textgene_names(textgenes, text|text)
    gene_stats = []
    for _, row in df_filtered.iterrows():
        gene_names = str(row['gene_names']).split('|')
        products = str(row['products']).split('|')
        microbes = str(row['microbes']).split('|')
        for i, gene_name in enumerate(gene_names):
            if gene_name != 'N/A':
                product = products[i] if i < len(products) else 'N/A'
                microbe = microbes[i] if i < len(microbes) else 'N/A'
                gene_stats.append({
                    'gene_name': gene_name,
                    'product': product,
                    'microbe': microbe
                })
                if not gene_stats:
                    return pd.DataFrame()
                df_stats = pd.DataFrame(gene_stats)
                # textgenestatisticspeptide count
                gene_summary = df_stats.groupby(['gene_name', 'product', 'microbe']).size().reset_index(name='peptide_count')
                gene_summary = gene_summary.sort_values('peptide_count', ascending=False)
                return gene_summary


# ============================================
# text
# ============================================
def main():
    logger.info("="*60)
    logger.info("startfiltertextpeptide (text - textgeneannotation)")
    logger.info("="*60)
    logger.info(f"\ninput directory: {csv_dir}")
    logger.info(f"output directory: {output_base_dir}")
    logger.info(f"\nImprovement notes:")
    logger.info(f" New new: textgeneannotationinformation(gene_name, producttext)")
    logger.info(f" New new: generategenetextpeptidestatistics")
    logger.info(f" New new: textpeptidestextgenes")
    # findall _all_records.csv file
    pattern = "*_fully_contained_all_records.csv"
    csv_files = list(Path(csv_dir).glob(pattern))
    if not csv_files:
        logger.error(f"\nnot foundmatchedfile: {pattern}")
        return
    logger.info(f"\nfound {len(csv_files)} samplesfile")
    # statistics
    stats = {
        'total_samples': len(csv_files),
        'processed_samples': 0,
        'classI_generated': 0,
        'classII_generated': 0,
        'failed_samples': []
    }
    # processtextsamples
    for csv_file in sorted(csv_files):
        try:
            filter_peptides_from_r_output(
                str(csv_file),
                classI_dir,
                classII_dir,
                pHLA_bare_dir,
                output_base_dir
)
stats['processed_samples'] += 1
# checkOutput files
sample_name = extract_sample_name(csv_file.name)
classI_output = os.path.join(output_base_dir, f"{sample_name}_classI_filtered.tsv")
classII_output = os.path.join(output_base_dir, f"{sample_name}_classII_filtered.tsv")
if os.path.exists(classI_output):
    stats['classI_generated'] += 1
    if os.path.exists(classII_output):
        stats['classII_generated'] += 1
        except Exception as e:
            logger.error(f"Process samplefailed: {csv_file.name}")
            logger.error(f"error: {str(e)}")
            stats['failed_samples'].append(csv_file.name)
            # textstatisticstext
            logger.info("\n" + "="*60)
            logger.info("Processing completed - statisticstext")
            logger.info("="*60)
            logger.info(f"\ntextsample count: {stats['total_samples']}")
            logger.info(f"processed successfully: {stats['processed_samples']}")
            logger.info(f"generate Class I results: {stats['classI_generated']}")
            logger.info(f"generate Class II results: {stats['classII_generated']}")
            if stats['failed_samples']:
                logger.info(f"\nprocessfailedsample ({len(stats['failed_samples'])}):")
                for sample in stats['failed_samples']:
                    logger.info(f" - {sample}")
                    # savestatistics report
                    summary_file = os.path.join(output_base_dir, "filtering_summary.txt")
                    with open(summary_file, 'w', encoding='utf-8') as f:
                        f.write("textpeptidefilterstatistics report(text - textgeneannotation)\n")
                        f.write("="*60 + "\n\n")
                        f.write(f"input directory: {csv_dir}\n")
                        f.write(f"output directory: {output_base_dir}\n\n")
                        f.write(f"textsample count: {stats['total_samples']}\n")
                        f.write(f"processed successfully: {stats['processed_samples']}\n")
                        f.write(f"generate Class I results: {stats['classI_generated']}\n")
                        f.write(f"generate Class II results: {stats['classII_generated']}\n")
                        if stats['failed_samples']:
                            f.write(f"\nprocessfailedsample:\n")
                            for sample in stats['failed_samples']:
                                f.write(f" - {sample}\n")
                                f.write("\nImprovement notes:\n")
                                f.write(" 1. Output filestextgeneannotation column: gene_names, products, microbes, locus_tags\n")
                                f.write(" 2. Generate gene-level statistics files: [sample]_[class]_gene_stats.tsv\n")
                                f.write(" 3. texttotextpeptidestextgenestext\n")
                                logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                                logger.info("\nallProcessing completed!")


if __name__ == "__main__":
    main()

# pan-cancer

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Multi-cancer antigen peptide filtering (improved version, includes gene annotation)
Filter corresponding MHC-binding peptide prediction results based on matched MTR reads
Filter through the source_ids -> Identity -> sseq -> pairs mapping relationship
Also attach gene annotation fields such as gene_name and product

Compatible with the _fully_contained_all_records.csv output format from the R code (with gene annotation)

File naming convention: - MTRinput: {sample_name}_fully_contained_all_records.csv (text: SRR9623616_fully_contained_all_records.csv)
- pHLA_baredirectory: {prefix}.{sample_name}/ (text: BLCA.Normal.SRR9623616/)
- MHCfile: {prefix}.{sample_name}.pHLA_I.tsv.gz (text: BLCA.Normal.SRR9623616.pHLA_I.tsv.gz)
Sample names are extracted from the field after the final dot
"""

import os
import pandas as pd
import gzip
from pathlib import Path
import logging
from multiprocessing import Pool
import time
from collections import defaultdict

# ============================================
# Configuration parameters
# ============================================
# cancer-type list
CANCERS = ["CRC", "BRCA", "CESC", "LIHC", "OSCC", "LUNG", "PAAD", "STAD", "BLCA"]

# response type
RESPONSE_TYPES = ["Tumor"] # can add "R", "NR", and similar groups

# base path
BASE_DIR = "/path/to/project/results_V3/cancers_V3.1"
PEPTIDE_BASE = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_public"

# textrowsprocesstext
N_PROCESSES = 9 # 9 cancer types can be processed in parallel

# ============================================
# Logging setup function
# ============================================
def setup_logger(cancer_type, response_type, output_dir):
    """foreachcancer type-response typecreatetextlogger"""
    log_file = os.path.join(output_dir, f"{cancer_type}_{response_type}_processing.log")
    logger = logging.getLogger(f"{cancer_type}_{response_type}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    # file handler
    fh = logging.FileHandler(log_file, mode='w', encoding='utf-8')
    fh.setLevel(logging.INFO)
    # console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    # format
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# ============================================
# Helper functions
# ============================================
def read_gzip_csv(file_path, **kwargs):
    """read.gztextCSV/TSVfile"""
    if file_path.endswith('.gz'):
        with gzip.open(file_path, 'rt') as f:
            return pd.read_csv(f, **kwargs)
    else:
        return pd.read_csv(file_path, **kwargs)


def extract_sample_name(filename):
    """
    Extract the sample name from the filename output by the R code
    format: {sample_name}_fully_contained_all_records.csv
    """
    base_name = Path(filename).stem
    sample_name = base_name.replace("_fully_contained_all_records", "")
    # onlytext(textfirst)
    sample_name = sample_name.split('_')[0]
    return sample_name


def load_mapping_tables(sample_name, logger):
    """
    loadmappingtext
    mappingtext:     1. id2seq.tsv: Identity -> sseq
    2. sseq2pairs.tsv: sseq -> pairs (textreads ID)
    return: (id2seq_df, sseq2pairs_df) or (None, None) textfiletextexists
    """
    # textmatchedfindtextsampletextdirectory
    # format: BLCA.Normal.SRR9623616 or CRC.R.B7
    bare_base = os.path.join(PEPTIDE_BASE, "pHLA_bare")
    # findmatchedsampletextdirectory(sampletextafter)
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(bare_base).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        logger.error(f" text: {sample_dir_pattern} in {bare_base}")
        return None, None
    bare_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {bare_dir.name}")
    # findmappingfile(textmatched)
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(bare_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(bare_dir.glob(sseq2pairs_pattern))
    if not id2seq_files:
        logger.error(f" not found id2seq file: {id2seq_pattern}")
        return None, None
    if not sseq2pairs_files:
        logger.error(f" not found sseq2pairs file: {sseq2pairs_pattern}")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        # readmappingtext
        id2seq_df = read_gzip_csv(str(id2seq_file), sep='\t')
        sseq2pairs_df = read_gzip_csv(str(sseq2pairs_file), sep='\t', header=None,             names=['sseq', 'pairs'])
        # check required columns
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns (Identity, sseq)")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df):
    """
    text source_ids mappingto pairs
    mappingtext:
    source_ids -> Identity (in id2seq) -> sseq -> pairs (in sseq2pairs)
    """
    # Step 1: source_ids -> sseq (passed Identity column)
    id2seq_dict = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
    # text source_ids mappingto sseq
    sseqs = [id2seq_dict.get(source_id) for source_id in source_ids_list]
    sseqs = [s for s in sseqs if pd.notna(s)]
    if not sseqs:
        return []
    # Step 2: sseq -> pairs
    sseq2pairs_dict = dict(zip(sseq2pairs_df['sseq'], sseq2pairs_df['pairs']))
    # text sseq mappingto pairs
    pairs_list = [sseq2pairs_dict.get(sseq) for sseq in sseqs]
    pairs_list = [p for p in pairs_list if pd.notna(p)]
    return pairs_list


def build_read_to_gene_mapping(df_csv, logger):
    """
    build reads ID togeneinformationmapping
    text:
    df_csv: RoutputCSVdata(textQNAMEandgeneannotation column)
    return:
    dict: {QNAME: {gene_info}}
    """
    read_to_gene = {}
    # textavailablegeneannotation column
    gene_cols = ['gene_id', 'gene_name', 'locus_tag', 'product', 'biotype', 'strand', 'gene_length']
    available_cols = [col for col in gene_cols if col in df_csv.columns]
    if not available_cols:
        logger.warning(" warning: texttogeneannotation column, textonlytextQNAME")
        return {}
    logger.info(f" texttogeneannotation column: {', '.join(available_cols)}")
    # foreachQNAMEsavegeneinformation
    for _, row in df_csv.iterrows():
        qname = row['QNAME']
        # extractgeneinformation
        gene_info = {}
        for col in available_cols:
            value = row.get(col)
            if pd.notna(value) and value != 'N/A':
                gene_info[col] = value
                # textwithgenetextinformation, textsave
                if 'RNAME' in df_csv.columns:
                    gene_info['chromosome'] = row.get('RNAME')
                    if 'Region_Start' in df_csv.columns and 'Region_End' in df_csv.columns:
                        gene_info['region'] = f"{row.get('Region_Start')}-{row.get('Region_End')}"
                        if 'microbe' in df_csv.columns or 'microbes' in df_csv.columns:
                            microbe_col = 'microbe' if 'microbe' in df_csv.columns else 'microbes'
                            gene_info['microbe'] = row.get(microbe_col)
                            # textreadtextexists, textinformation
                            if qname in read_to_gene:
                                existing = read_to_gene[qname]
                                if isinstance(existing, dict):
                                    read_to_gene[qname] = [existing, gene_info]
                                else:
                                    read_to_gene[qname].append(gene_info)
                            else:
                                read_to_gene[qname] = gene_info
                                logger.info(f" buildtext {len(read_to_gene)} readgenemapping")
                                return read_to_gene


def generate_gene_statistics(df_filtered, mhc_class):
    """
    generategenetextpeptidestatistics
    """
    if df_filtered.empty or 'gene_names' not in df_filtered.columns:
        return pd.DataFrame()
    # textgene_names(textgenes, text|text)
    gene_stats = []
    for _, row in df_filtered.iterrows():
        gene_names = str(row['gene_names']).split('|')
        products = str(row['products']).split('|')
        microbes = str(row['microbes']).split('|')
        for i, gene_name in enumerate(gene_names):
            if gene_name != 'N/A':
                product = products[i] if i < len(products) else 'N/A'
                microbe = microbes[i] if i < len(microbes) else 'N/A'
                gene_stats.append({
                    'gene_name': gene_name,
                    'product': product,
                    'microbe': microbe
                })
                if not gene_stats:
                    return pd.DataFrame()
                df_stats = pd.DataFrame(gene_stats)
                # textgenestatisticspeptide count
                gene_summary = df_stats.groupby(['gene_name', 'product', 'microbe']).size().reset_index(name='peptide_count')
                gene_summary = gene_summary.sort_values('peptide_count', ascending=False)
                return gene_summary


# ============================================
# processMHCtextfile
# ============================================
def process_mhc_file(mhc_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,
    output_dir, sample_name, mhc_class, cancer_type, response_type, logger):
"""
processMHCtextfile, passedmappingtextfiltermatchedpeptide, textgeneannotation
"""
if not os.path.exists(mhc_file):
    logger.warning(f" {mhc_class} filetextexists: {mhc_file}")
    return 0
    try:
        # readMHCtextresults
        df_mhc = read_gzip_csv(mhc_file, sep='\t')
        # check required columns
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return 0
        logger.info(f" MHCtextrecord: {len(df_mhc)} text")
        # text reads_ids textfor set textfind
        reads_ids_set = set(reads_ids)
        # recordmatcheddetailedinformationandgeneinformation
        gene_annotations = []
        def check_match_and_get_gene_info(source_ids):
            """
            checkmatchedtextgeneinformation
            """
            if pd.isna(source_ids):
                return False, None
            source_ids_list = str(source_ids).split(',')
            pairs_list = map_source_ids_to_pairs(source_ids_list, id2seq_df, sseq2pairs_df)
            matched_genes = []
            for pairs in pairs_list:
                if pd.isna(pairs):
                    continue
                pairs_str = str(pairs)
                # foundmatchedreadstextgeneinformation
                for read_id in reads_ids_set:
                    if read_id in pairs_str:
                        # textreadtextgeneinformation
                        if read_id in read_to_gene:
                            gene_info = read_to_gene[read_id]
                            if isinstance(gene_info, list):
                                matched_genes.extend(gene_info)
                            else:
                                matched_genes.append(gene_info)
                                if matched_genes:
                                    return True, matched_genes
                                else:
                                    return False, None
                                # filtermatchedrecordtextgeneinformation
                                logger.info(f" startfiltermatchedpeptidetextgeneinformation...")
                                filtered_rows = []
                                for idx, row in df_mhc.iterrows():
                                    is_match, gene_infos = check_match_and_get_gene_info(row['source_ids'])
                                    if is_match:
                                        filtered_rows.append(row)
                                        # extractgeneannotationinformation
                                        if gene_infos:
                                            gene_names = set()
                                            products = set()
                                            microbes = set()
                                            locus_tags = set()
                                            for gene_info in gene_infos:
                                                if 'gene_name' in gene_info:
                                                    gene_names.add(str(gene_info['gene_name']))
                                                    if 'product' in gene_info:
                                                        products.add(str(gene_info['product']))
                                                        if 'microbe' in gene_info:
                                                            microbes.add(str(gene_info['microbe']))
                                                            if 'locus_tag' in gene_info:
                                                                locus_tags.add(str(gene_info['locus_tag']))
                                                                gene_annotations.append({
                                                                    'gene_names': '|'.join(sorted(gene_names)) if gene_names else 'N/A',
                                                                    'products': '|'.join(sorted(products)) if products else 'N/A',
                                                                    'microbes': '|'.join(sorted(microbes)) if microbes else 'N/A',
                                                                    'locus_tags': '|'.join(sorted(locus_tags)) if locus_tags else 'N/A'
                                                                })
                                                            else:
                                                                gene_annotations.append({
                                                                    'gene_names': 'N/A',
                                                                    'products': 'N/A',
                                                                    'microbes': 'N/A',
                                                                    'locus_tags': 'N/A'
                                                                })
                                                                if not filtered_rows:
                                                                    logger.warning(f" nomatchedrecord, textgenerateOutput files")
                                                                    return 0
                                                                df_mhc_filtered = pd.DataFrame(filtered_rows)
                                                                n_filtered = len(df_mhc_filtered)
                                                                n_total = len(df_mhc)
                                                                percentage = (n_filtered/n_total*100) if n_total > 0 else 0
                                                                logger.info(f" filterresults: {n_filtered}/{n_total} records ({percentage:.2f}%)")
                                                                # textgeneannotation column
                                                                df_gene_anno = pd.DataFrame(gene_annotations)
                                                                df_mhc_filtered = pd.concat([df_mhc_filtered.reset_index(drop=True),                                                                     df_gene_anno], axis=1)
                                                                # textinformationcolumn
                                                                df_mhc_filtered['sample'] = sample_name
                                                                df_mhc_filtered['cancer_type'] = cancer_type
                                                                df_mhc_filtered['response_type'] = response_type
                                                                # textcolumncolumntext, textgeneannotationtextfirsttext
                                                                cols = df_mhc_filtered.columns.tolist()
                                                                priority_cols = ['sample', 'cancer_type', 'response_type',                                                                     'gene_names', 'products', 'microbes', 'locus_tags']
                                                                other_cols = [c for c in cols if c not in priority_cols]
                                                                df_mhc_filtered = df_mhc_filtered[priority_cols + other_cols]
                                                                # savefilterresults
                                                                output_file = os.path.join(output_dir, f"{sample_name}_{mhc_class}_filtered.tsv")
                                                                df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
                                                                logger.info(f" savedtext: {os.path.basename(output_file)}")
                                                                # generategenetextstatistics
                                                                gene_stats = generate_gene_statistics(df_mhc_filtered, mhc_class)
                                                                if not gene_stats.empty:
                                                                    gene_stats_file = os.path.join(output_dir,                                                                         f"{sample_name}_{mhc_class}_gene_stats.tsv")
                                                                    gene_stats.to_csv(gene_stats_file, sep='\t', index=False)
                                                                    logger.info(f" genestatisticssaved: {os.path.basename(gene_stats_file)}")
                                                                    # texttopgene
                                                                    logger.info(f" Top 5 gene(textpeptide count):")
                                                                    for i, row in gene_stats.head(5).iterrows():
                                                                        logger.info(f" {i+1}. {row['gene_name']}: {row['peptide_count']} peptide ({row['product']})")
                                                                        return n_filtered
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return 0


# ============================================
# processtextsamples
# ============================================
def process_sample(csv_file, cancer_type, response_type, output_dir, logger):
    """
    fromRtextoutput _all_records.csv fileinextractreads IDandgeneinformation, filterpeptidetextresults
    """
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\n{'='*60}")
    logger.info(f"Process sample: {sample_name}")
    logger.info(f"inputfile: {file_name}")
    logger.info(f"{'='*60}")
    try:
        # readRtextoutputCSVfile
        df_csv = pd.read_csv(csv_file, sep=',')
        logger.info(f" filecolumn: {df_csv.columns.tolist()}")
        logger.info(f" total records: {len(df_csv)}")
        # check required columns
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return None
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        # statisticsmicrobeinformation
        if 'microbe' in df_csv.columns or 'microbes' in df_csv.columns:
            microbe_col = 'microbe' if 'microbe' in df_csv.columns else 'microbes'
            microbes = df_csv[microbe_col].unique()
            logger.info(f" text {len(microbes)} microbes")
            if len(microbes) <= 10:
                logger.info(f" microbe list: {', '.join(str(m) for m in microbes)}")
                if n_reads == 0:
                    logger.warning(f" warning: textwithreads ID, skip")
                    return None
                # build read -> geneinformation mapping
                logger.info(f"\n buildreadstogenemappingtext...")
                read_to_gene = build_read_to_gene_mapping(df_csv, logger)
                # loadmappingtext
                logger.info(f"\n loadmappingtext...")
                id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, logger)
                if id2seq_df is None or sseq2pairs_df is None:
                    logger.error(f" notextloadmappingtext, skipsample {sample_name}")
                    return None
                # process MHC Class I
                logger.info(f"\n find MHC Class I textresults...")
                # textmatchedfindfile: *.{sample_name}.pHLA_I.tsv.gz
                classI_dir = os.path.join(PEPTIDE_BASE, "BindLevel", "HLAI")
                classI_pattern = f"*.{sample_name}.pHLA_I.tsv*"
                classI_files = list(Path(classI_dir).glob(classI_pattern))
                if classI_files:
                    classI_file = str(classI_files[0])
                    logger.info(f" found Class I file: {os.path.basename(classI_file)}")
                    n_classI = process_mhc_file(
                        classI_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,
                        output_dir, sample_name, "classI", cancer_type, response_type, logger
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    n_classI = 0
    # process MHC Class II
    logger.info(f"\n find MHC Class II textresults...")
    # textmatchedfindfile: *.{sample_name}.pHLA_II.tsv.gz
    classII_dir = os.path.join(PEPTIDE_BASE, "BindLevel", "HLAII")
    classII_pattern = f"*.{sample_name}.pHLA_II.tsv*"
    classII_files = list(Path(classII_dir).glob(classII_pattern))
    if classII_files:
        classII_file = str(classII_files[0])
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        n_classII = process_mhc_file(
            classII_file, reads_ids, read_to_gene, id2seq_df, sseq2pairs_df,
            output_dir, sample_name, "classII", cancer_type, response_type, logger
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    n_classII = 0
    logger.info(f" sample {sample_name} Processing completed")
    return {
        'sample': sample_name,
        'classI': n_classI,
        'classII': n_classII
    }
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return None


# ============================================
# processtextcancer type-response typetext
# ============================================
def process_cancer_response(args):
    """processtextcancer type-response typeallsample"""
    cancer_type, response_type = args
    # output directory
    output_dir = os.path.join(
        BASE_DIR, cancer_type, "06.antigen", "06.2.MTR_peptides", response_type
)
os.makedirs(output_dir, exist_ok=True)
# textlogger
logger = setup_logger(cancer_type, response_type, output_dir)
logger.info("="*70)
logger.info(f"startprocess: {cancer_type} - {response_type}")
logger.info("="*70)
# MTRfiledirectory
mtr_dir = os.path.join(
    BASE_DIR, cancer_type, "05.MTR", "05.5.MTR_sample_fully_contained", response_type
)
# checkinput directorytextexists
if not os.path.exists(mtr_dir):
    logger.warning(f"input directorytextexists, skip: {mtr_dir}")
    return {f"{cancer_type}_{response_type}": None}
    # findall _fully_contained_all_records.csv file
    pattern = "*_fully_contained_all_records.csv"
    csv_files = list(Path(mtr_dir).glob(pattern))
    if not csv_files:
        logger.warning(f"not foundmatchedfile: {pattern}")
        return {f"{cancer_type}_{response_type}": None}
    logger.info(f"found {len(csv_files)} samplesfile")
    # statistics
    stats = {
        'samples': 0,
        'classI': 0,
        'classII': 0,
        'total_classI_records': 0,
        'total_classII_records': 0,
        'failed_samples': []
    }
    # processtextsamples
    for csv_file in sorted(csv_files):
        result = process_sample(str(csv_file), cancer_type, response_type, output_dir, logger)
        if result:
            stats['samples'] += 1
            if result['classI'] > 0:
                stats['classI'] += 1
                stats['total_classI_records'] += result['classI']
                if result['classII'] > 0:
                    stats['classII'] += 1
                    stats['total_classII_records'] += result['classII']
                else:
                    stats['failed_samples'].append(os.path.basename(csv_file))
                    # savestatistics report
                    logger.info("\n" + "="*70)
                    logger.info(f"{cancer_type}-{response_type} Processing completed - statisticstext")
                    logger.info("="*70)
                    logger.info(f"Process sampletext: {stats['samples']}")
                    logger.info(f"generate Class I resultsfile: {stats['classI']}")
                    logger.info(f"generate Class II resultsfile: {stats['classII']}")
                    logger.info(f"Class I total records: {stats['total_classI_records']}")
                    logger.info(f"Class II total records: {stats['total_classII_records']}")
                    if stats['failed_samples']:
                        logger.info(f"\nprocessfailedsample ({len(stats['failed_samples'])}):")
                        for sample in stats['failed_samples']:
                            logger.info(f" - {sample}")
                            # savestatisticsfile
                            summary_file = os.path.join(output_dir, f"{cancer_type}_{response_type}_summary.txt")
                            with open(summary_file, 'w', encoding='utf-8') as f:
                                f.write(f"{cancer_type}-{response_type} textpeptidefilterstatistics report\n")
                                f.write("="*70 + "\n\n")
                                f.write(f"Process sampletext: {stats['samples']}\n")
                                f.write(f"generate Class I resultsfile: {stats['classI']}\n")
                                f.write(f"generate Class II resultsfile: {stats['classII']}\n")
                                f.write(f"Class I total records: {stats['total_classI_records']}\n")
                                f.write(f"Class II total records: {stats['total_classII_records']}\n")
                                if stats['failed_samples']:
                                    f.write(f"\nprocessfailedsample:\n")
                                    for sample in stats['failed_samples']:
                                        f.write(f" - {sample}\n")
                                        f.write("\nImprovement notes:\n")
                                        f.write(" 1. Output filestextgeneannotation column: gene_names, products, microbes, locus_tags\n")
                                        f.write(" 2. Generate gene-level statistics files: [sample]_[class]_gene_stats.tsv\n")
                                        f.write(" 3. texttotextpeptidestextgenestext\n")
                                        logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                                        return {f"{cancer_type}_{response_type}": stats}


# ============================================
# text
# ============================================
def main():
    """text: textrowsprocessallcancer type-response typetext"""
    print("="*70)
    print("textcancer typetextpeptidefiltertext(text - textgeneannotation)")
    print("="*70)
    print(f"cancer typecount: {len(CANCERS)}")
    print(f"cancer-type list: {', '.join(CANCERS)}")
    print(f"response type: {', '.join(RESPONSE_TYPES)}")
    print(f"textrowstext: {N_PROCESSES}")
    print("\nImprovement notes:")
    print(" New new: textgeneannotationinformation(gene_name, producttext)")
    print(" New new: generategenetextpeptidestatistics")
    print(" New new: textpeptidestextgenes")
    print("="*70)
    start_time = time.time()
    # buildallcancer type-response typetext
    tasks = [(cancer, response) for cancer in CANCERS for response in RESPONSE_TYPES]
    print(f"\ntext: {len(tasks)}")
    # textrowsprocessallcancer type-response typetext
    with Pool(processes=N_PROCESSES) as pool:
        results = pool.map(process_cancer_response, tasks)
        # Summarize all results
        print("\n" + "="*70)
        print("allcancer typeProcessing completed - Overall statistics")
        print("="*70)
        total_stats = {}
        for result in results:
            total_stats.update(result)
            # saveOverall statistics
            global_summary_file = os.path.join(BASE_DIR, "all_cancers_summary_with_genes.txt")
            with open(global_summary_file, 'w', encoding='utf-8') as f:
                f.write("Overall multi-cancer antigen peptide filtering report (with gene annotation)\n")
                f.write("="*70 + "\n\n")
                for cancer_type in CANCERS:
                    f.write(f"\n{cancer_type}:\n")
                    f.write("-"*50 + "\n")
                    for response_type in RESPONSE_TYPES:
                        key = f"{cancer_type}_{response_type}"
                        if key in total_stats and total_stats[key]:
                            stats = total_stats[key]
                            f.write(f" {response_type}:\n")
                            f.write(f" Process sampletext: {stats['samples']}\n")
                            f.write(f" generate Class I results: {stats['classI']}\n")
                            f.write(f" generate Class II results: {stats['classII']}\n")
                            f.write(f" Class I total records: {stats['total_classI_records']}\n")
                            f.write(f" Class II total records: {stats['total_classII_records']}\n")
                        else:
                            f.write(f" {response_type}: nodataorprocessfailed\n")
                            f.write("\nImprovement notes:\n")
                            f.write(" 1. textRtextoutput _fully_contained_all_records.csv format\n")
                            f.write(" 2. Output filestextgeneannotation column: gene_names, products, microbes, locus_tags\n")
                            f.write(" 3. Generate gene-level statistics files: [sample]_[class]_gene_stats.tsv\n")
                            f.write(" 4. textcancer typetextrowsprocess\n")
                            elapsed_time = time.time() - start_time
                            print(f"\ntext: {elapsed_time:.2f} text ({elapsed_time/60:.2f} text)")
                            print(f"Overall statisticssavedtext: {global_summary_file}")
                            print("\nProcessing completed!")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Multi-cancer antigen peptide filtering (performance-optimized version)
Main optimizations: 1. Prebuild index structures to avoid repeated computation
2. Use vectorized operations instead of row-by-row iteration
3. Optimize string matching logic
4. Add progress display
5. Memory optimization
"""

import os
import pandas as pd
import gzip
from pathlib import Path
import logging
from multiprocessing import Pool
import time
from collections import defaultdict
import numpy as np
from tqdm import tqdm
import re

# ============================================
# Configuration parameters
# ============================================
CANCERS = ["CRC", "BRCA", "CESC", "LIHC", "OSCC", "LUNG", "PAAD", "STAD", "BLCA"]
RESPONSE_TYPES = ["Tumor"]
BASE_DIR = "/path/to/project/results_V3/cancers_V3.1"
PEPTIDE_BASE = "/path/to/project/data_V3/peptides/MiTRI_reads_peptides_public"
N_PROCESSES = 9

# ============================================
# Logging setup
# ============================================
def setup_logger(cancer_type, response_type, output_dir):
    """foreachcancer type-response typecreatetextlogger"""
    log_file = os.path.join(output_dir, f"{cancer_type}_{response_type}_processing.log")
    logger = logging.getLogger(f"{cancer_type}_{response_type}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fh = logging.FileHandler(log_file, mode='w', encoding='utf-8')
    fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# ============================================
# textHelper functions
# ============================================
def read_gzip_csv(file_path, **kwargs):
    """read.gztextCSV/TSVfile"""
    if file_path.endswith('.gz'):
        with gzip.open(file_path, 'rt') as f:
            return pd.read_csv(f, **kwargs)
    else:
        return pd.read_csv(file_path, **kwargs)


def extract_sample_name(filename):
    """Extract the sample name from the filename output by the R code"""
    base_name = Path(filename).stem
    sample_name = base_name.replace("_fully_contained_all_records", "")
    sample_name = sample_name.split('_')[0]
    return sample_name


def load_mapping_tables(sample_name, logger):
    """loadmappingtext"""
    bare_base = os.path.join(PEPTIDE_BASE, "pHLA_bare")
    sample_dir_pattern = f"*.{sample_name}"
    sample_dirs = list(Path(bare_base).glob(sample_dir_pattern))
    if not sample_dirs:
        logger.error(f" not foundsample {sample_name} pHLA_bare directory")
        return None, None
    bare_dir = sample_dirs[0]
    logger.info(f" using mapping-file directory: {bare_dir.name}")
    id2seq_pattern = f"*.{sample_name}.id2seq.tsv*"
    sseq2pairs_pattern = f"*.{sample_name}.sseq2pairs.tsv*"
    id2seq_files = list(bare_dir.glob(id2seq_pattern))
    sseq2pairs_files = list(bare_dir.glob(sseq2pairs_pattern))
    if not id2seq_files or not sseq2pairs_files:
        logger.error(f" not foundmappingfile")
        return None, None
    id2seq_file = id2seq_files[0]
    sseq2pairs_file = sseq2pairs_files[0]
    logger.info(f" load id2seq: {id2seq_file.name}")
    logger.info(f" load sseq2pairs: {sseq2pairs_file.name}")
    try:
        id2seq_df = read_gzip_csv(str(id2seq_file), sep='\t')
        sseq2pairs_df = read_gzip_csv(str(sseq2pairs_file), sep='\t', header=None,             names=['sseq', 'pairs'])
        if 'Identity' not in id2seq_df.columns or 'sseq' not in id2seq_df.columns:
            logger.error(f" id2seq file is missing required columns")
            return None, None
        logger.info(f" id2seq: {len(id2seq_df)} records")
        logger.info(f" sseq2pairs: {len(sseq2pairs_df)} records")
        return id2seq_df, sseq2pairs_df
    except Exception as e:
        logger.error(f" loadmappingfiletext: {str(e)}")
        return None, None


def build_optimized_index(id2seq_df, sseq2pairs_df, reads_ids_set, logger):
    """
    Fast text: textbuildsource_idtoreadstextmappingtext
    textprocesstextMHCrecordtextcalculate
    """
    logger.info(" buildtext...")
    start = time.time()
    # Step 1: build sseq -> reads mapping
    sseq_to_reads = {}
    for _, row in tqdm(sseq2pairs_df.iterrows(), total=len(sseq2pairs_df),         desc=" process sseq2pairs", disable=True):
    sseq = row['sseq']
    pairs_str = str(row['pairs'])
    # frompairsinextractallreads ID(text)
    matched_reads = []
    for read_id in reads_ids_set:
        # text: textmatched, textmatched
        if re.search(rf'\b{re.escape(read_id)}\b', pairs_str):
            matched_reads.append(read_id)
            if matched_reads:
                sseq_to_reads[sseq] = matched_reads
                logger.info(f" sseq_to_reads: {len(sseq_to_reads)} textwithtextmapping")
                # Step 2: build source_id -> reads mapping
                source_id_to_reads = {}
                # textbuild Identity -> sseq mapping
                id_to_sseq = dict(zip(id2seq_df['Identity'], id2seq_df['sseq']))
                for identity, sseq in tqdm(id_to_sseq.items(), desc=" build source_id index", disable=True):
                    if pd.notna(sseq) and sseq in sseq_to_reads:
                        source_id_to_reads[identity] = sseq_to_reads[sseq]
                        elapsed = time.time() - start
                        logger.info(f" source_id_to_reads: {len(source_id_to_reads)} textwithtextmapping")
                        logger.info(f" textbuildcompleted, text: {elapsed:.2f} text")
                        return source_id_to_reads


def build_read_to_gene_mapping(df_csv, logger):
    """build reads ID togeneinformationmapping"""
    read_to_gene = {}
    gene_cols = ['gene_id', 'gene_name', 'locus_tag', 'product', 'biotype', 'strand', 'gene_length']
    available_cols = [col for col in gene_cols if col in df_csv.columns]
    if not available_cols:
        logger.warning(" warning: texttogeneannotation column")
        return {}
    logger.info(f" texttogeneannotation column: {', '.join(available_cols)}")
    # textprocessgeneinformation
    for _, row in df_csv.iterrows():
        qname = row['QNAME']
        gene_info = {}
        for col in available_cols:
            value = row.get(col)
            if pd.notna(value) and value != 'N/A':
                gene_info[col] = value
                if 'RNAME' in df_csv.columns:
                    gene_info['chromosome'] = row.get('RNAME')
                    if 'Region_Start' in df_csv.columns and 'Region_End' in df_csv.columns:
                        gene_info['region'] = f"{row.get('Region_Start')}-{row.get('Region_End')}"
                        if 'microbe' in df_csv.columns or 'microbes' in df_csv.columns:
                            microbe_col = 'microbe' if 'microbe' in df_csv.columns else 'microbes'
                            gene_info['microbe'] = row.get(microbe_col)
                            if qname in read_to_gene:
                                existing = read_to_gene[qname]
                                if isinstance(existing, dict):
                                    read_to_gene[qname] = [existing, gene_info]
                                else:
                                    read_to_gene[qname].append(gene_info)
                            else:
                                read_to_gene[qname] = gene_info
                                logger.info(f" buildtext {len(read_to_gene)} readgenemapping")
                                return read_to_gene


def generate_gene_statistics(df_filtered, mhc_class):
    """generategenetextpeptidestatistics"""
    if df_filtered.empty or 'gene_names' not in df_filtered.columns:
        return pd.DataFrame()
    gene_stats = []
    for _, row in df_filtered.iterrows():
        gene_names = str(row['gene_names']).split('|')
        products = str(row['products']).split('|')
        microbes = str(row['microbes']).split('|')
        for i, gene_name in enumerate(gene_names):
            if gene_name != 'N/A':
                product = products[i] if i < len(products) else 'N/A'
                microbe = microbes[i] if i < len(microbes) else 'N/A'
                gene_stats.append({
                    'gene_name': gene_name,
                    'product': product,
                    'microbe': microbe
                })
                if not gene_stats:
                    return pd.DataFrame()
                df_stats = pd.DataFrame(gene_stats)
                gene_summary = df_stats.groupby(['gene_name', 'product', 'microbe']).size().reset_index(name='peptide_count')
                gene_summary = gene_summary.sort_values('peptide_count', ascending=False)
                return gene_summary


# ============================================
# textMHCfileprocess
# ============================================
def process_mhc_file_optimized(mhc_file, reads_ids_set, read_to_gene,     source_id_to_reads, output_dir, sample_name,     mhc_class, cancer_type, response_type, logger):
"""
Fast text: textfilter
"""
if not os.path.exists(mhc_file):
    logger.warning(f" {mhc_class} filetextexists: {mhc_file}")
    return 0
    try:
        logger.info(f" read {mhc_class} file...")
        df_mhc = read_gzip_csv(mhc_file, sep='\t')
        if 'source_ids' not in df_mhc.columns:
            logger.error(f" {mhc_class} filetext source_ids column")
            return 0
        logger.info(f" MHCtextrecord: {len(df_mhc)} text")
        # Fast textprocess: textcheckmatched
        logger.info(f" startfiltermatchedpeptide...")
        matched_indices = []
        gene_annotations = []
        # texttqdmtext
        for idx, source_ids in tqdm(enumerate(df_mhc['source_ids']),             total=len(df_mhc),             desc=f" filter{mhc_class}"):
        if pd.isna(source_ids):
            continue
        # textsource_ids
        source_ids_list = str(source_ids).split(',')
        # Fast textfind
        matched_reads = set()
        for source_id in source_ids_list:
            if source_id in source_id_to_reads:
                matched_reads.update(source_id_to_reads[source_id])
                # checktextwithmatchedreads
                if matched_reads:
                    matched_indices.append(idx)
                    # textgeneinformation
                    gene_infos = []
                    for read_id in matched_reads:
                        if read_id in read_to_gene:
                            gene_info = read_to_gene[read_id]
                            if isinstance(gene_info, list):
                                gene_infos.extend(gene_info)
                            else:
                                gene_infos.append(gene_info)
                                # extractgeneannotation
                                if gene_infos:
                                    gene_names = set()
                                    products = set()
                                    microbes = set()
                                    locus_tags = set()
                                    for gene_info in gene_infos:
                                        if 'gene_name' in gene_info:
                                            gene_names.add(str(gene_info['gene_name']))
                                            if 'product' in gene_info:
                                                products.add(str(gene_info['product']))
                                                if 'microbe' in gene_info:
                                                    microbes.add(str(gene_info['microbe']))
                                                    if 'locus_tag' in gene_info:
                                                        locus_tags.add(str(gene_info['locus_tag']))
                                                        gene_annotations.append({
                                                            'gene_names': '|'.join(sorted(gene_names)) if gene_names else 'N/A',
                                                            'products': '|'.join(sorted(products)) if products else 'N/A',
                                                            'microbes': '|'.join(sorted(microbes)) if microbes else 'N/A',
                                                            'locus_tags': '|'.join(sorted(locus_tags)) if locus_tags else 'N/A'
                                                        })
                                                    else:
                                                        gene_annotations.append({
                                                            'gene_names': 'N/A',
                                                            'products': 'N/A',
                                                            'microbes': 'N/A',
                                                            'locus_tags': 'N/A'
                                                        })
                                                        if not matched_indices:
                                                            logger.warning(f" nomatchedrecord, textgenerateOutput files")
                                                            return 0
                                                        # Fast textiloctextextract, textrowstext
                                                        df_mhc_filtered = df_mhc.iloc[matched_indices].copy()
                                                        n_filtered = len(df_mhc_filtered)
                                                        n_total = len(df_mhc)
                                                        percentage = (n_filtered/n_total*100) if n_total > 0 else 0
                                                        logger.info(f" filterresults: {n_filtered}/{n_total} records ({percentage:.2f}%)")
                                                        # textgeneannotation column
                                                        df_gene_anno = pd.DataFrame(gene_annotations)
                                                        df_mhc_filtered.reset_index(drop=True, inplace=True)
                                                        df_mhc_filtered = pd.concat([df_mhc_filtered, df_gene_anno], axis=1)
                                                        # textinformationcolumn
                                                        df_mhc_filtered['sample'] = sample_name
                                                        df_mhc_filtered['cancer_type'] = cancer_type
                                                        df_mhc_filtered['response_type'] = response_type
                                                        # textcolumncolumntext
                                                        cols = df_mhc_filtered.columns.tolist()
                                                        priority_cols = ['sample', 'cancer_type', 'response_type',                                                             'gene_names', 'products', 'microbes', 'locus_tags']
                                                        other_cols = [c for c in cols if c not in priority_cols]
                                                        df_mhc_filtered = df_mhc_filtered[priority_cols + other_cols]
                                                        # savefilterresults
                                                        output_file = os.path.join(output_dir, f"{sample_name}_{mhc_class}_filtered.tsv")
                                                        df_mhc_filtered.to_csv(output_file, sep='\t', index=False)
                                                        logger.info(f" savedtext: {os.path.basename(output_file)}")
                                                        # generategenetextstatistics
                                                        gene_stats = generate_gene_statistics(df_mhc_filtered, mhc_class)
                                                        if not gene_stats.empty:
                                                            gene_stats_file = os.path.join(output_dir,                                                                 f"{sample_name}_{mhc_class}_gene_stats.tsv")
                                                            gene_stats.to_csv(gene_stats_file, sep='\t', index=False)
                                                            logger.info(f" genestatisticssaved: {os.path.basename(gene_stats_file)}")
                                                            logger.info(f" Top 5 gene(textpeptide count):")
                                                            for i, row in gene_stats.head(5).iterrows():
                                                                logger.info(f" {i+1}. {row['gene_name']}: {row['peptide_count']} peptide ({row['product']})")
                                                                return n_filtered
    except Exception as e:
        logger.error(f" process {mhc_class} filetext: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return 0


# ============================================
# processtextsamples
# ============================================
def process_sample(csv_file, cancer_type, response_type, output_dir, logger):
    """textsampleprocess"""
    file_name = os.path.basename(csv_file)
    sample_name = extract_sample_name(file_name)
    logger.info(f"\n{'='*60}")
    logger.info(f"Process sample: {sample_name}")
    logger.info(f"inputfile: {file_name}")
    logger.info(f"{'='*60}")
    try:
        # readCSVfile
        df_csv = pd.read_csv(csv_file, sep=',')
        logger.info(f" filecolumn: {df_csv.columns.tolist()}")
        logger.info(f" total records: {len(df_csv)}")
        if 'QNAME' not in df_csv.columns:
            logger.error(f" error: {csv_file} text QNAME column")
            return None
        # extractuniquereads ID
        reads_ids = df_csv['QNAME'].unique()
        reads_ids_set = set(reads_ids) # Fast Use a set to speed up lookup
        n_reads = len(reads_ids)
        logger.info(f" extractto {n_reads} uniquereads ID")
        # statisticsmicrobeinformation
        if 'microbe' in df_csv.columns or 'microbes' in df_csv.columns:
            microbe_col = 'microbe' if 'microbe' in df_csv.columns else 'microbes'
            microbes = df_csv[microbe_col].unique()
            logger.info(f" text {len(microbes)} microbes")
            if len(microbes) <= 10:
                logger.info(f" microbe list: {', '.join(str(m) for m in microbes)}")
                if n_reads == 0:
                    logger.warning(f" warning: textwithreads ID, skip")
                    return None
                # build read -> geneinformation mapping
                logger.info(f"\n buildreadstogenemappingtext...")
                read_to_gene = build_read_to_gene_mapping(df_csv, logger)
                # loadmappingtext
                logger.info(f"\n loadmappingtext...")
                id2seq_df, sseq2pairs_df = load_mapping_tables(sample_name, logger)
                if id2seq_df is None or sseq2pairs_df is None:
                    logger.error(f" notextloadmappingtext, skipsample {sample_name}")
                    return None
                # Fast text: textbuildtext
                source_id_to_reads = build_optimized_index(id2seq_df, sseq2pairs_df,                     reads_ids_set, logger)
                # process MHC Class I
                logger.info(f"\n find MHC Class I textresults...")
                classI_dir = os.path.join(PEPTIDE_BASE, "BindLevel", "HLAI")
                classI_pattern = f"*.{sample_name}.pHLA_I.tsv*"
                classI_files = list(Path(classI_dir).glob(classI_pattern))
                if classI_files:
                    classI_file = str(classI_files[0])
                    logger.info(f" found Class I file: {os.path.basename(classI_file)}")
                    n_classI = process_mhc_file_optimized(
                        classI_file, reads_ids_set, read_to_gene, source_id_to_reads,
                        output_dir, sample_name, "classI", cancer_type, response_type, logger
)
else:
    logger.warning(f" Class I filetextexists, sample: {sample_name}")
    n_classI = 0
    # process MHC Class II
    logger.info(f"\n find MHC Class II textresults...")
    classII_dir = os.path.join(PEPTIDE_BASE, "BindLevel", "HLAII")
    classII_pattern = f"*.{sample_name}.pHLA_II.tsv*"
    classII_files = list(Path(classII_dir).glob(classII_pattern))
    if classII_files:
        classII_file = str(classII_files[0])
        logger.info(f" found Class II file: {os.path.basename(classII_file)}")
        n_classII = process_mhc_file_optimized(
            classII_file, reads_ids_set, read_to_gene, source_id_to_reads,
            output_dir, sample_name, "classII", cancer_type, response_type, logger
)
else:
    logger.warning(f" Class II filetextexists, sample: {sample_name}")
    n_classII = 0
    logger.info(f" sample {sample_name} Processing completed")
    return {
        'sample': sample_name,
        'classI': n_classI,
        'classII': n_classII
    }
    except Exception as e:
        logger.error(f" process {csv_file} text: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return None


# ============================================
# processtextcancer type-response typetext
# ============================================
def process_cancer_response(args):
    """processtextcancer type-response typeallsample"""
    cancer_type, response_type = args
    output_dir = os.path.join(
        BASE_DIR, cancer_type, "06.antigen", "06.2.MTR_peptides", response_type
)
os.makedirs(output_dir, exist_ok=True)
logger = setup_logger(cancer_type, response_type, output_dir)
logger.info("="*70)
logger.info(f"startprocess: {cancer_type} - {response_type}")
logger.info("="*70)
mtr_dir = os.path.join(
    BASE_DIR, cancer_type, "05.MTR", "05.5.MTR_sample_fully_contained", response_type
)
if not os.path.exists(mtr_dir):
    logger.warning(f"input directorytextexists, skip: {mtr_dir}")
    return {f"{cancer_type}_{response_type}": None}
    pattern = "*_fully_contained_all_records.csv"
    csv_files = list(Path(mtr_dir).glob(pattern))
    if not csv_files:
        logger.warning(f"not foundmatchedfile: {pattern}")
        return {f"{cancer_type}_{response_type}": None}
    logger.info(f"found {len(csv_files)} samplesfile")
    stats = {
        'samples': 0,
        'classI': 0,
        'classII': 0,
        'total_classI_records': 0,
        'total_classII_records': 0,
        'failed_samples': []
    }
    # processtextsamples
    for csv_file in sorted(csv_files):
        result = process_sample(str(csv_file), cancer_type, response_type, output_dir, logger)
        if result:
            stats['samples'] += 1
            if result['classI'] > 0:
                stats['classI'] += 1
                stats['total_classI_records'] += result['classI']
                if result['classII'] > 0:
                    stats['classII'] += 1
                    stats['total_classII_records'] += result['classII']
                else:
                    stats['failed_samples'].append(os.path.basename(csv_file))
                    # savestatistics report
                    logger.info("\n" + "="*70)
                    logger.info(f"{cancer_type}-{response_type} Processing completed - statisticstext")
                    logger.info("="*70)
                    logger.info(f"Process sampletext: {stats['samples']}")
                    logger.info(f"generate Class I resultsfile: {stats['classI']}")
                    logger.info(f"generate Class II resultsfile: {stats['classII']}")
                    logger.info(f"Class I total records: {stats['total_classI_records']}")
                    logger.info(f"Class II total records: {stats['total_classII_records']}")
                    if stats['failed_samples']:
                        logger.info(f"\nprocessfailedsample ({len(stats['failed_samples'])}):")
                        for sample in stats['failed_samples']:
                            logger.info(f" - {sample}")
                            summary_file = os.path.join(output_dir, f"{cancer_type}_{response_type}_summary.txt")
                            with open(summary_file, 'w', encoding='utf-8') as f:
                                f.write(f"{cancer_type}-{response_type} textpeptidefilterstatistics report\n")
                                f.write("="*70 + "\n\n")
                                f.write(f"Process sampletext: {stats['samples']}\n")
                                f.write(f"generate Class I resultsfile: {stats['classI']}\n")
                                f.write(f"generate Class II resultsfile: {stats['classII']}\n")
                                f.write(f"Class I total records: {stats['total_classI_records']}\n")
                                f.write(f"Class II total records: {stats['total_classII_records']}\n")
                                if stats['failed_samples']:
                                    f.write(f"\nprocessfailedsample:\n")
                                    for sample in stats['failed_samples']:
                                        f.write(f" - {sample}\n")
                                        f.write("\nPerformance optimization notes:\n")
                                        f.write(" 1. New Prebuilt index: direct source_id -> reads mapping\n")
                                        f.write(" 2. New Vectorized operations: batch processing instead of row iteration\n")
                                        f.write(" 3. New textmatched: textmatched\n")
                                        f.write(" 4. New Progress display: tqdm shows processing progress\n")
                                        f.write(" 5. New Memory optimization: textdata\n")
                                        logger.info(f"\nstatistics reportsavedtext: {summary_file}")
                                        return {f"{cancer_type}_{response_type}": stats}


# ============================================
# text
# ============================================
def main():
    """text: textrowsprocessallcancer type-response typetext"""
    print("="*70)
    print("textcancer typetextpeptidefiltertext(text)")
    print("="*70)
    print(f"cancer typecount: {len(CANCERS)}")
    print(f"cancer-type list: {', '.join(CANCERS)}")
    print(f"response type: {', '.join(RESPONSE_TYPES)}")
    print(f"textrowstext: {N_PROCESSES}")
    print("\nFast text:")
    print(" New Prebuild index structures to avoid repeated computation")
    print(" New textrowstext")
    print(" New Optimize string matching logic")
    print(" New Add progress display(tqdm)")
    print(" New Memory optimization, text")
    print("="*70)
    start_time = time.time()
    tasks = [(cancer, response) for cancer in CANCERS for response in RESPONSE_TYPES]
    print(f"\ntext: {len(tasks)}")
    # textrowsprocess
    with Pool(processes=N_PROCESSES) as pool:
        results = pool.map(process_cancer_response, tasks)
        # textresults
        print("\n" + "="*70)
        print("allcancer typeProcessing completed - Overall statistics")
        print("="*70)
        total_stats = {}
        for result in results:
            total_stats.update(result)
            # saveOverall statistics
            global_summary_file = os.path.join(BASE_DIR, "all_cancers_summary_optimized.txt")
            with open(global_summary_file, 'w', encoding='utf-8') as f:
                f.write("Overall multi-cancer antigen peptide filtering report (performance optimized)\n")
                f.write("="*70 + "\n\n")
                for cancer_type in CANCERS:
                    f.write(f"\n{cancer_type}:\n")
                    f.write("-"*50 + "\n")
                    for response_type in RESPONSE_TYPES:
                        key = f"{cancer_type}_{response_type}"
                        if key in total_stats and total_stats[key]:
                            stats = total_stats[key]
                            f.write(f" {response_type}:\n")
                            f.write(f" Process sampletext: {stats['samples']}\n")
                            f.write(f" generate Class I results: {stats['classI']}\n")
                            f.write(f" generate Class II results: {stats['classII']}\n")
                            f.write(f" Class I total records: {stats['total_classI_records']}\n")
                            f.write(f" Class II total records: {stats['total_classII_records']}\n")
                        else:
                            f.write(f" {response_type}: nodataorprocessfailed\n")
                            f.write("\nPerformance optimization notes:\n")
                            f.write(" 1. New Prebuilt index: direct source_id -> reads mapping, textcalculate\n")
                            f.write(" 2. New text: textpandastextprocesstextrowstext\n")
                            f.write(" 3. New textmatched: textmatchedtext\n")
                            f.write(" 4. New Progress display: tqdm shows processing progress\n")
                            f.write(" 5. New Memory optimization: text\n")
                            elapsed_time = time.time() - start_time
                            print(f"\ntext: {elapsed_time:.2f} text ({elapsed_time/60:.2f} text)")
                            print(f"Overall statisticssavedtext: {global_summary_file}")
                            print("\nOK Processing completed!")


if __name__ == "__main__":
    main()